<a href="https://colab.research.google.com/github/samer-glitch/TADP-Cluster-Computing/blob/main/TADP_CIFAR_10_Results.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow scikit-learn matplotlib pandas numpy psutil

In [2]:
# ======================================================================================
# TADP v16.7 — REVIEWER-ALIGNED TRUSTWORTHY DATA PREPARATION EXPERIMENT CORE
# ======================================================================================
# Design guarantees:
#   1) GLOBAL holdout is created BEFORE client partitioning.
#   2) Only TRAIN is distributed to clients.
#   3) Preprocessing parameters/vocabularies use TRAIN only.
#   4) DQ, GE and TADP governance use TRAIN only.
#   5) Held-out TEST never affects preprocessing, governance, class weights,
#      client selection, model initialization, training, or matched-control budgets.
#   6) TEST is used only after training for final evaluation.
#
# v16.7 governance:
#   - 28 factors across 6 dimensions:
#       dim1=4, dim2(DQ)=8, dim3=4, dim4=3, dim5=5, dim6=4.
#   - Added Data Collection / Acquisition Lineage (dim1).
#   - Added Structural / Constraint Integrity (dim2).
#   - Documentary evidence is generated at the individual factor level.
#   - DQ evidence is machine-measured from client TRAIN partitions only.
#   - HPS remains client-specific and uses the six policy dimension weights.
#   - WAC is NOT client-specific.
#   - Each factor has a declared minimum adequate rubric rank.
#   - A domain WAC is derived once:
#         WAC_d = mean(minimum adequate ranks in dimension d) / 5
#         WAC_domain = equal mean of the six WAC_d values.
#   - Every averaged dimension must be >= 2.5/5 or the client is auto-rejected.
#   - HPS < 3.0 -> AUTO_REJECT.
#   - 3.0 <= HPS < 3.5 -> AUTOMATED REVIEW using Critical WAC_i.
#   - Review accepts iff Critical WAC_i >= the domain Global Critical WAC.
#   - HPS >= 3.5 -> DIRECT AUTO_ACCEPT when every critical factor meets its own adequacy minimum.
#   - Human reviewers verify evidence only; admission is server-automated.
#
# GX Core comparator:
#   - Separate from HPS/TADP.
#   - Uses REAL Great Expectations GX Core validation on TRAIN-only client data.
#   - Covers schema, volume, missingness, uniqueness/duplicates, cardinality,
#     type/value consistency, distribution, and integrity/constraints.
#   - GX validates data; a separate matched-K ranking selects the same number
#     of clients as TADP-VR for a controlled downstream comparison.
#
# Runtime/reporting:
#   - Every completed FL round prints an immediate ROUND x/y DONE message.
#   - Every completed scenario prints all predictive and operational metrics.
#   - Scenario checkpoints support restart/resume.
#   - Final result ZIP downloads automatically in Google Colab.
# ======================================================================================

import os
import sys
import gc
import math
import time
import json
import random
import hashlib
import threading
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


# ======================================================================================
# POLICY CONSTANTS — v16.7
# ======================================================================================

GOOD_CUT = 3.0
HIGH_CUT = 3.5
MAX_FACTOR_SCORE = 5.0
GE_ACCEPT_COUNT = 6

# ----------------------------------------------------------------------
# v16.7 FINAL FULLY AUTOMATED ADMISSION POLICY
# ----------------------------------------------------------------------
# Human reviewers verify supporting evidence uploaded through the questionnaire.
# They do NOT make the admission decision. Once verified factor scores are
# available, the server applies this policy automatically.
#
# 1) EVERY averaged HPS dimension must be >= 2.5/5.
#    Any dimension < 2.5 -> AUTO-REJECT.
#
# 2) HPS must be >= 3.0.
#    HPS < 3.0 -> AUTO-REJECT.
#
# 3) DIRECT AUTO-ACCEPT:
#    HPS >= 3.5 AND every critical factor independently meets its own
#    factor-specific minimum adequacy requirement.
#
# 4) AUTOMATED REVIEW:
#    - all clients with 3.0 <= HPS < 3.5; and
#    - high-HPS clients that fail one or more individual critical-factor
#      adequacy requirements.
#
# 5) REVIEW RESOLUTION:
#    Critical WAC_i = mean(actual critical-factor scores / 5)
#    Global Critical WAC = mean(policy adequacy minima / 5 for same factors)
#
#    Critical WAC_i >= Global Critical WAC -> ACCEPT AFTER REVIEW
#    Critical WAC_i <  Global Critical WAC -> AUTO-REJECT
#
# Thus Direct Auto-Accept is strict at the individual critical-factor level,
# whereas Review intentionally allows controlled compensation across the
# critical subset.
DIMENSION_MIN_FLOOR = 2.5

CRITICAL_FACTORS_BY_DOMAIN = {
    "healthcare": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim5": [
            "regulation_coverage",
            "consent_ethics",
            "sensitivity_classification",
        ],
        "dim6": [
            "user_agreements",
        ],
    },
    "cifar10": {
        "dim1": [
            "data_controller",
            "data_collection_lineage",
        ],
        "dim6": [
            "license_terms",
            "user_agreements",
        ],
    },
}

WEIGHTS_PSCORE_DEFAULT = {
    "dim1": 0.25,  # Source Reliability
    "dim2": 0.15,  # Data Quality and Health
    "dim3": 0.10,  # Documentation Practices
    "dim4": 0.10,  # Timeliness and Refresh Rate
    "dim5": 0.30,  # Regulatory / Compliance Alignment
    "dim6": 0.10,  # Context / Usage Constraints
}

DIMENSION_NAMES = {
    "dim1": "Source Reliability",
    "dim2": "Data Quality and Health",
    "dim3": "Documentation Practices",
    "dim4": "Timeliness and Refresh Rate",
    "dim5": "Regulatory and Compliance Alignment",
    "dim6": "Context and Usage Constraints",
}

# v16.7: two reviewer-driven additions:
#   dim1: data_collection_lineage
#   dim2: structural_constraint_integrity
#
# Total = 4 + 8 + 4 + 3 + 5 + 4 = 28 factors.
FACTOR_NAMES = {
    "dim1": [
        "source_reputation",
        "data_controller",
        "data_objective",
        "data_collection_lineage",
    ],
    "dim2": [
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ],
    "dim3": [
        "data_dictionary",
        "version_logs",
        "collection_protocol",
        "definition_updates",
    ],
    "dim4": [
        "data_freshness",
        "scheduled_refresh",
        "retention_clarity",
    ],
    "dim5": [
        "regulation_coverage",
        "consent_ethics",
        "geo_restrictions",
        "sensitivity_classification",
        "audits",
    ],
    "dim6": [
        "license_terms",
        "ethical_reviews",
        "redistribution",
        "user_agreements",
    ],
}

DOCUMENTARY_DIMS = ("dim1", "dim3", "dim4", "dim5", "dim6")

# ----------------------------------------------------------------------
# COMPLETE 0--5 RUBRIC DESCRIPTORS
# ----------------------------------------------------------------------
# These descriptors reproduce the Appendix-A semantics and add the two
# v16.7 factors explicitly. Controlled evidence is sampled at the factor
# level; HPS is never generated directly.
RUBRIC_DESCRIPTORS = {
    "dim1": {
        "source_reputation": {
            0: "No info",
            1: "Poor",
            2: "Limited evidence",
            3: "Average, partially trusted",
            4: "Well-documented, reliable",
            5: "Highly reputable, verified",
        },
        "data_controller": {
            0: "No documented controller",
            1: "Unclear",
            2: "Partially clear",
            3: "Moderately clear",
            4: "Mostly clear",
            5: "Fully documented",
        },
        "data_objective": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General but unclear",
            4: "Mostly explicit",
            5: "Fully explicit, justified",
        },
        "data_collection_lineage": {
            0: "Collection origin unknown",
            1: "Informal or unverifiable origin",
            2: "Partially documented acquisition path",
            3: "Documented acquisition with limited traceability",
            4: "Well-documented and traceable acquisition path",
            5: "Fully source-linked, versioned, and auditable lineage",
        },
    },
    "dim2": {
        "completeness": {
            0: ">50% missing",
            1: "20-50% missing",
            2: "10-20% missing",
            3: "5-10% missing",
            4: "1-5% missing",
            5: "<1% missing",
        },
        "duplication_rate": {
            0: ">20% duplicates",
            1: "10-20% duplicates",
            2: "5-10% duplicates",
            3: "2-5% duplicates",
            4: "1-2% duplicates",
            5: "<1% duplicates",
        },
        "value_validity_error_rate": {
            0: ">15% invalid/error values",
            1: "10-15% invalid/error values",
            2: "5-10% invalid/error values",
            3: "2-5% invalid/error values",
            4: "1-2% invalid/error values",
            5: "<1% invalid/error values",
        },
        "type_consistency": {
            0: "Highly inconsistent",
            1: "Frequent type inconsistency",
            2: "Moderate type inconsistency",
            3: "Minor type inconsistency",
            4: "Rare type inconsistency",
            5: "Fully consistent",
        },
        "label_integrity": {
            0: ">10% missing/invalid/known erroneous labels",
            1: "5-10% missing/invalid/known erroneous labels",
            2: "2-5% missing/invalid/known erroneous labels",
            3: "1-2% missing/invalid/known erroneous labels",
            4: "0.1-1% missing/invalid/known erroneous labels",
            5: "<=0.1% missing/invalid/known erroneous labels",
        },
        "feature_distribution_consistency": {
            0: "JSD >0.20",
            1: "JSD 0.10-0.20",
            2: "JSD 0.05-0.10",
            3: "JSD 0.025-0.05",
            4: "JSD 0.01-0.025",
            5: "JSD <=0.01",
        },
        "feature_category_coverage": {
            0: "<50% reference support represented",
            1: "50-65% reference support represented",
            2: "65-75% reference support represented",
            3: "75-82.5% reference support represented",
            4: "82.5-90% reference support represented",
            5: ">=90% reference support represented",
        },
        "structural_constraint_integrity": {
            0: ">10% records/structures violate required constraints",
            1: "5-10% violate required constraints",
            2: "2-5% violate required constraints",
            3: "1-2% violate required constraints",
            4: "0.1-1% violate required constraints",
            5: "<=0.1% violate required constraints",
        },
    },
    "dim3": {
        "data_dictionary": {
            0: "None",
            1: "Minimal outline",
            2: "Partial coverage",
            3: "Moderate coverage",
            4: "Near-complete",
            5: "Fully detailed",
        },
        "version_logs": {
            0: "None",
            1: "Minimal logs",
            2: "Occasional logs",
            3: "Regular logs",
            4: "Near-complete",
            5: "Full version history",
        },
        "collection_protocol": {
            0: "None",
            1: "Vague",
            2: "Partial",
            3: "General methods",
            4: "Well-defined",
            5: "Fully transparent",
        },
        "definition_updates": {
            0: "None",
            1: "Rarely updated",
            2: "Occasional updates",
            3: "Regular but basic",
            4: "Frequent",
            5: "Real-time, documented",
        },
    },
    "dim4": {
        "data_freshness": {
            0: ">5 years old",
            1: "2-5 years old",
            2: "1-2 years old",
            3: "6-12 months old",
            4: "1-6 months old",
            5: "Real-time/current",
        },
        "scheduled_refresh": {
            0: "Never",
            1: "Irregular",
            2: "Annual",
            3: "Quarterly",
            4: "Monthly",
            5: "Daily/real-time",
        },
        "retention_clarity": {
            0: "None",
            1: "Minimal",
            2: "Basic guidelines",
            3: "Moderate clarity",
            4: "High clarity",
            5: "Fully documented",
        },
    },
    "dim5": {
        "regulation_coverage": {
            0: "None",
            1: "Minimal",
            2: "Partial",
            3: "Moderate",
            4: "Comprehensive but dated",
            5: "Fully documented/current",
        },
        "consent_ethics": {
            0: "None",
            1: "Minimal record",
            2: "Partial consent/ethics evidence",
            3: "Moderate logs",
            4: "Substantial",
            5: "Fully documented",
        },
        "geo_restrictions": {
            0: "None",
            1: "Basic mention",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully documented",
        },
        "sensitivity_classification": {
            0: "None",
            1: "Basic flagging",
            2: "Partial",
            3: "Moderate",
            4: "Near-complete",
            5: "Fully classified",
        },
        "audits": {
            0: "None",
            1: "Internal only",
            2: "Basic certification",
            3: "Occasional audit",
            4: "Recent audit",
            5: "Regular external audits",
        },
    },
    "dim6": {
        "license_terms": {
            0: "None",
            1: "Vague",
            2: "Basic",
            3: "Clear",
            4: "Detailed",
            5: "Industry-compliant",
        },
        "ethical_reviews": {
            0: "None",
            1: "Informal approval",
            2: "Partial",
            3: "Moderate",
            4: "Well-documented",
            5: "Certified",
        },
        "redistribution": {
            0: "No policy",
            1: "Unclear",
            2: "Partial",
            3: "Clear",
            4: "Detailed",
            5: "Fully compliant",
        },
        "user_agreements": {
            0: "Non-compliant",
            1: "Minimal adherence",
            2: "Partial",
            3: "Mostly compliant",
            4: "Fully compliant",
            5: "Audited compliance",
        },
    },
}

# ----------------------------------------------------------------------
# FACTOR-SPECIFIC ADEQUACY POLICY
# ----------------------------------------------------------------------
# The minimum adequate rank is derived factor-by-factor from the wording of
# the Appendix-A rubric. It is NOT learned from model/test outcomes.
#
# Healthcare is the primary policy. CIFAR-10 uses the same reference ranks
# for cross-domain comparability; its DQ factors are measured with image-
# specific checks, while documentary dimensions remain controlled evidence.
FACTOR_ADEQUACY_MIN_HEALTHCARE = {
    "dim1": {
        "source_reputation": 4,
        "data_controller": 4,
        "data_objective": 4,
        "data_collection_lineage": 4,
    },
    "dim2": {
        "completeness": 3,
        "duplication_rate": 3,
        "value_validity_error_rate": 3,
        "type_consistency": 3,
        "label_integrity": 3,
        "feature_distribution_consistency": 3,
        "feature_category_coverage": 3,
        "structural_constraint_integrity": 3,
    },
    "dim3": {
        "data_dictionary": 3,
        "version_logs": 3,
        "collection_protocol": 4,
        "definition_updates": 3,
    },
    "dim4": {
        "data_freshness": 3,
        "scheduled_refresh": 3,
        "retention_clarity": 4,
    },
    "dim5": {
        "regulation_coverage": 3,
        "consent_ethics": 3,
        "geo_restrictions": 3,
        "sensitivity_classification": 3,
        "audits": 4,
    },
    "dim6": {
        "license_terms": 3,
        "ethical_reviews": 4,
        "redistribution": 3,
        "user_agreements": 4,
    },
}

FACTOR_ADEQUACY_MIN_CIFAR10 = {
    dim: dict(values)
    for dim, values in FACTOR_ADEQUACY_MIN_HEALTHCARE.items()
}

def derive_domain_wac(
    factor_minima: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Derive the domain policy WAC from factor-specific minimum adequate ranks.

    WAC_d = mean_k(adequate_rank_dk / 5)
    WAC_domain = equal mean across the six dimension WAC_d values.

    IMPORTANT:
      WAC is a DOMAIN POLICY value, not a client-specific score.
    """
    dimension_wac = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(factor_minima[dim][factor])
            for factor in FACTOR_NAMES[dim]
        ]
        dimension_wac[dim] = float(np.mean(vals) / MAX_FACTOR_SCORE)
    global_wac = float(np.mean(list(dimension_wac.values())))
    return dimension_wac, global_wac


HEALTHCARE_DIMENSION_WAC, WAC_HEALTHCARE = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_HEALTHCARE
)
CIFAR10_DIMENSION_WAC, WAC_CIFAR10 = derive_domain_wac(
    FACTOR_ADEQUACY_MIN_CIFAR10
)

DOMAIN_FACTOR_MINIMA = {
    "healthcare": FACTOR_ADEQUACY_MIN_HEALTHCARE,
    "cifar10": FACTOR_ADEQUACY_MIN_CIFAR10,
}
DOMAIN_DIMENSION_WAC = {
    "healthcare": HEALTHCARE_DIMENSION_WAC,
    "cifar10": CIFAR10_DIMENSION_WAC,
}
DOMAIN_GLOBAL_WAC = {
    "healthcare": WAC_HEALTHCARE,
    "cifar10": WAC_CIFAR10,
}

DQ_RAW_METRIC_BY_FACTOR = {
    "completeness": "missing_fraction",
    "duplication_rate": "duplicate_fraction",
    "value_validity_error_rate": "error_fraction",
    "type_consistency": "type_inconsistency_fraction",
    "label_integrity": "invalid_label_fraction",
    "feature_distribution_consistency": "max_jsd",
    "feature_category_coverage": "mean_category_coverage",
    "structural_constraint_integrity": "structural_violation_fraction",
}

assert abs(sum(WEIGHTS_PSCORE_DEFAULT.values()) - 1.0) < 1e-12
assert sum(len(v) for v in FACTOR_NAMES.values()) == 28
assert len(FACTOR_NAMES["dim1"]) == 4
assert len(FACTOR_NAMES["dim2"]) == 8
assert abs(WAC_HEALTHCARE - 0.6761111111111111) < 1e-12

# ======================================================================================
# GENERAL UTILITIES
# ======================================================================================

def seed_everything(seed: int):
    seed = int(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


def sha256_bytes(x: bytes) -> str:
    return hashlib.sha256(x).hexdigest()


def sha256_array(x: np.ndarray) -> str:
    x = np.asarray(x)
    return sha256_bytes(np.ascontiguousarray(x).view(np.uint8).tobytes())


def sha256_weights(weights: List[np.ndarray]) -> str:
    h = hashlib.sha256()
    for w in weights:
        a = np.ascontiguousarray(np.asarray(w))
        h.update(str(a.shape).encode())
        h.update(a.view(np.uint8).tobytes())
    return h.hexdigest()


def _process_rss_mb() -> float:
    try:
        import psutil
        return float(psutil.Process(os.getpid()).memory_info().rss / (1024 ** 2))
    except Exception:
        pass
    try:
        with open("/proc/self/statm", "r", encoding="utf-8") as f:
            pages = int(f.read().split()[1])
        return float(pages * int(os.sysconf("SC_PAGE_SIZE")) / (1024 ** 2))
    except Exception:
        pass
    try:
        import resource
        x = float(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss)
        return x / (1024 ** 2) if sys.platform == "darwin" else x / 1024.0
    except Exception:
        return 0.0


class RAMMonitor:
    def __init__(self, interval_s: float = 0.05):
        self.interval_s = float(interval_s)
        self.start_mb = 0.0
        self.end_mb = 0.0
        self.peak_mb = 0.0
        self._stop = threading.Event()
        self._thread = None

    def _loop(self):
        while not self._stop.wait(self.interval_s):
            self.peak_mb = max(self.peak_mb, _process_rss_mb())

    def start(self):
        self.start_mb = _process_rss_mb()
        self.peak_mb = self.start_mb
        self._stop.clear()
        self._thread = threading.Thread(target=self._loop, daemon=True)
        self._thread.start()
        return self

    def stop(self) -> Dict[str, float]:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=1.0)
        self.end_mb = _process_rss_mb()
        self.peak_mb = max(self.peak_mb, self.start_mb, self.end_mb)
        return {
            "ram_start_mb": float(self.start_mb),
            "ram_end_mb": float(self.end_mb),
            "ram_peak_mb": float(self.peak_mb),
            "ram_delta_mb": float(max(0.0, self.peak_mb - self.start_mb)),
            "ram_mb": float(self.peak_mb),
        }


def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)
    return Path(path)


def append_csv(row: Dict[str, Any], path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([row]).to_csv(
        path, mode="a", header=not path.exists(), index=False
    )


def summarize_runs(perf: pd.DataFrame) -> pd.DataFrame:
    if perf.empty:
        return pd.DataFrame()
    metrics = [
        c for c in [
            "accuracy", "precision_macro", "recall_macro", "f1_macro",
            "roc_auc_ovr_macro", "runtime_s", "energy_wh", "communication_mb",
            "optimizer_steps", "participants", "ram_peak_mb", "ram_delta_mb"
        ] if c in perf.columns
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": len(d)}
        for m in metrics:
            vals = pd.to_numeric(d[m], errors="coerce")
            row[f"{m}_mean"] = float(vals.mean())
            row[f"{m}_sd"] = float(vals.std(ddof=1)) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows)


def package_and_download_results(output_dir: Path, label: str) -> Path:
    output_dir = Path(output_dir)
    manifest = []
    for p in sorted(output_dir.rglob("*")):
        if p.is_file():
            manifest.append({
                "relative_path": str(p.relative_to(output_dir)),
                "bytes": int(p.stat().st_size),
                "sha256": hashlib.sha256(p.read_bytes()).hexdigest(),
            })
    pd.DataFrame(manifest).to_csv(
        output_dir / "RESULTS_FILE_MANIFEST.csv", index=False
    )

    zip_path = output_dir.parent / f"{output_dir.name}_RESULTS.zip"
    if zip_path.exists():
        zip_path.unlink()

    with zipfile.ZipFile(
        zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6
    ) as zf:
        for p in sorted(output_dir.rglob("*")):
            if p.is_file():
                zf.write(p, arcname=str(p.relative_to(output_dir)))

    print("\n" + "=" * 100)
    print(f"{label} COMPLETE")
    print(f"Results ZIP: {zip_path}")
    print(f"ZIP size: {zip_path.stat().st_size / (1024**2):.2f} MB")
    print("=" * 100)

    try:
        from google.colab import files as colab_files
        print("Starting automatic download to your laptop...")
        colab_files.download(str(zip_path))
    except Exception as exc:
        print("Automatic Colab download unavailable.")
        print(f"ZIP remains at: {zip_path}")
        print(f"Reason: {exc}")

    return zip_path


# ======================================================================================
# CONTROLLED DOCUMENTARY EVIDENCE — v16.7
# ======================================================================================

def rubric_descriptor(dim: str, factor: str, score: float) -> str:
    s = int(np.clip(np.rint(float(score)), 0, 5))
    return str(RUBRIC_DESCRIPTORS[dim][factor][s])


def generate_controlled_documentary_evidence(
    client_ids: List[str],
    evidence_seed: int,
    factor_minima: Dict[str, Dict[str, float]],
    domain: str,
) -> Tuple[Dict[str, Dict[str, Dict[str, float]]], pd.DataFrame]:
    """
    Generate controlled documentary evidence at the INDIVIDUAL FACTOR level.

    v16.7 uses PRE-SPECIFIED governance archetypes so the controlled experiment
    contains all three decision outcomes needed to validate the policy:

      - DIRECT_STRONG:
          designed to satisfy the strict Direct Auto-Accept route;
      - REVIEW_RECOVERABLE:
          borderline overall evidence, but sufficiently strong average evidence
          across the critical subset for Automated Review acceptance;
      - REVIEW_LIMITED:
          adequate enough to enter Automated Review, but insufficient average
          critical evidence for Review acceptance;
      - LOW_HPS_WEAK:
          every documentary dimension remains at/above the 2.5 dimension floor,
          but the overall HPS is expected to remain below 3.0;
      - DIMENSION_FLOOR_WEAK:
          contains a deliberately weak documentary dimension (<2.5).

    IMPORTANT SCIENTIFIC INTERPRETATION
    -----------------------------------
    These are controlled governance scenarios, not observed hospital-site
    provenance records and not estimates of real-world admission prevalence.
    The archetype composition is specified BEFORE model training and does not
    use predictive performance, test labels, or downstream model outcomes.

    DQ (dim2) is NEVER synthesized here; it remains measured from TRAIN data.
    The frozen evidence seed randomly assigns the pre-generated archetypes to
    client identities.
    """
    domain = str(domain).lower()
    if domain not in CRITICAL_FACTORS_BY_DOMAIN:
        raise ValueError(f"Unsupported domain: {domain!r}")

    n = len(client_ids)
    rng = np.random.default_rng(int(evidence_seed))

    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]
    critical_keys = {
        (dim, factor)
        for dim, factor_list in critical_policy.items()
        for factor in factor_list
    }

    def make_documentary_profile(role: str, variant: int):
        factors = {
            dim: {}
            for dim in DOCUMENTARY_DIMS
        }

        if role == "DIRECT_STRONG":
            # Strong but not uniformly perfect. Every factor is at least 4,
            # while some values reach 5 according to a deterministic variant.
            for dim in DOCUMENTARY_DIMS:
                for j, factor in enumerate(FACTOR_NAMES[dim]):
                    minimum = float(factor_minima[dim][factor])
                    base = max(4.0, minimum)
                    bonus = 1.0 if ((j + variant + len(dim)) % 3 == 0) else 0.0
                    factors[dim][factor] = float(min(5.0, base + bonus))

        elif role == "REVIEW_RECOVERABLE":
            # Start from moderate evidence: all documentary dimensions remain
            # safely above the 2.5 floor without making HPS automatically high.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            critical_sequence = [
                (dim, factor)
                for dim, factor_list in critical_policy.items()
                for factor in factor_list
            ]

            # Put critical factors at their policy adequacy references.
            for dim, factor in critical_sequence:
                factors[dim][factor] = float(
                    factor_minima[dim][factor]
                )

            # Intentionally allow ONE critical factor to fall one point below
            # its individual minimum, while another critical factor with room
            # is strengthened. This is exactly the compensatory situation that
            # Automated Review is intended to resolve.
            high_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) >= 4.0
            ]
            lower_min = [
                (dim, factor)
                for dim, factor in critical_sequence
                if float(factor_minima[dim][factor]) <= 3.0
            ]

            if high_min and lower_min:
                weak_key = high_min[variant % len(high_min)]
                strong_key = lower_min[variant % len(lower_min)]

                weak_min = float(
                    factor_minima[weak_key[0]][weak_key[1]]
                )
                strong_min = float(
                    factor_minima[strong_key[0]][strong_key[1]]
                )

                factors[weak_key[0]][weak_key[1]] = float(
                    max(0.0, weak_min - 1.0)
                )
                factors[strong_key[0]][strong_key[1]] = float(
                    min(5.0, strong_min + 2.0)
                )

            # Small documentary variation between the two recoverable bundles.
            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "REVIEW_LIMITED":
            # Preserve dimensions at/above 2.5 and keep HPS in/near the review
            # region, but make the average critical evidence too weak.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    if dim in critical_policy and (dim, factor) not in critical_keys:
                        factors[dim][factor] = 4.0
                    else:
                        factors[dim][factor] = 3.0

            for dim, factor_list in critical_policy.items():
                for factor in factor_list:
                    minimum = float(factor_minima[dim][factor])
                    factors[dim][factor] = float(
                        max(2.0, minimum - 1.0)
                    )

            if variant % 2 == 1:
                factors["dim3"][FACTOR_NAMES["dim3"][0]] = 4.0

        elif role == "LOW_HPS_WEAK":
            # Each documentary dimension averages >=2.5, but remains weak.
            # This isolates the HPS<3.0 rejection path from the dimension floor.
            for dim in DOCUMENTARY_DIMS:
                names = list(FACTOR_NAMES[dim])
                values = [2.0] * len(names)
                n_three = (len(names) + 1) // 2
                for j in range(n_three):
                    values[j] = 3.0

                for factor, value in zip(names, values):
                    factors[dim][factor] = float(value)

        elif role == "DIMENSION_FLOOR_WEAK":
            # General evidence is moderate, but Timeliness is deliberately below
            # the 2.5 dimension trustworthiness floor.
            for dim in DOCUMENTARY_DIMS:
                for factor in FACTOR_NAMES[dim]:
                    factors[dim][factor] = 3.0

            for factor in FACTOR_NAMES["dim4"]:
                factors["dim4"][factor] = 2.0

        else:
            raise ValueError(f"Unknown evidence role: {role!r}")

        return factors

    # For the fixed K=10 experiments, pre-specify a balanced governance
    # branch-coverage set:
    #   4 direct-strong
    #   2 review-recoverable
    #   2 review-limited
    #   1 low-HPS weak
    #   1 dimension-floor weak
    #
    # This is not a post-hoc selection by client identity. The ten bundles are
    # generated first and then randomly assigned to clients using evidence_seed.
    if n == 10:
        bundle_specs = [
            ("DIRECT_STRONG", 0),
            ("DIRECT_STRONG", 1),
            ("DIRECT_STRONG", 2),
            ("DIRECT_STRONG", 3),
            ("REVIEW_RECOVERABLE", 0),
            ("REVIEW_RECOVERABLE", 1),
            ("REVIEW_LIMITED", 0),
            ("REVIEW_LIMITED", 1),
            ("LOW_HPS_WEAK", 0),
            ("DIMENSION_FLOOR_WEAK", 0),
        ]
    else:
        # Generic fallback for non-10-client studies: cycle through the same
        # archetypes without conditioning on client data or model performance.
        archetypes = [
            "DIRECT_STRONG",
            "REVIEW_RECOVERABLE",
            "REVIEW_LIMITED",
            "LOW_HPS_WEAK",
            "DIMENSION_FLOOR_WEAK",
        ]
        bundle_specs = [
            (archetypes[j % len(archetypes)], j // len(archetypes))
            for j in range(n)
        ]

    bundles = []
    for b, (role, variant) in enumerate(bundle_specs):
        factors = make_documentary_profile(role, int(variant))

        bundles.append({
            "bundle_id": f"E{b+1:02d}",
            "profile": role,
            "scenario_role": role,
            "profile_variant": int(variant),
            "factors": factors,
        })

    # Randomly assign already-generated governance bundles to client identities.
    assignment = rng.permutation(n)

    evidence_by_client = {}
    rows = []

    for client_pos, cid in enumerate(client_ids):
        bundle = bundles[int(assignment[client_pos])]

        evidence_by_client[cid] = {
            dim: dict(bundle["factors"][dim])
            for dim in DOCUMENTARY_DIMS
        }

        for dim in DOCUMENTARY_DIMS:
            for factor, score in bundle["factors"][dim].items():
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client":
                        str(cid),
                    "bundle_id":
                        bundle["bundle_id"],
                    "evidence_profile":
                        bundle["profile"],
                    "scenario_role":
                        bundle["scenario_role"],
                    "profile_variant":
                        int(bundle["profile_variant"]),
                    "dimension":
                        dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor":
                        factor,
                    "evidence_source_type":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "rubric_score_0_5":
                        float(score),
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(float(score) >= min_rank),
                    "evidence_artifact_id": (
                        f"{cid}-{bundle['bundle_id']}-{dim}-{factor}"
                    ),
                    "validation_status":
                        "CONTROLLED_SCENARIO_EVIDENCE",
                    "evidence_seed":
                        int(evidence_seed),
                    "domain":
                        domain,
                })

    return evidence_by_client, pd.DataFrame(rows)

def dimension_scores_from_factors(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client-specific HPS dimension scores (0..5):
    mean of the observed factor rubric scores within each dimension.
    """
    out = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        out[dim] = float(np.mean(vals)) if vals else 0.0
    return out


def compute_hps(
    dimensions: Dict[str, float],
    weights: Dict[str, float] = WEIGHTS_PSCORE_DEFAULT,
) -> float:
    return float(
        sum(float(weights[d]) * float(dimensions[d]) for d in weights)
    )


def client_dimension_adequacy(
    factors: Dict[str, Dict[str, float]]
) -> Dict[str, float]:
    """
    Client ACHIEVED adequacy, not WAC.

    A_i,d = mean(client factor ranks in dimension d) / 5.
    """
    scores = {}
    for dim in FACTOR_NAMES:
        vals = [
            float(v)
            for v in factors.get(dim, {}).values()
            if np.isfinite(float(v))
        ]
        scores[dim] = (
            float(np.mean(vals) / MAX_FACTOR_SCORE)
            if vals else 0.0
        )
    return scores


def client_adequacy_score(
    factors: Dict[str, Dict[str, float]]
) -> Tuple[Dict[str, float], float]:
    """
    Legacy audit helper retained for backward comparability only.

    v16.7 does NOT use this value for admission. Review decisions use the
    client Critical WAC_i versus the domain Global Critical WAC.
    """
    by_dim = client_dimension_adequacy(factors)
    cas = float(np.mean(list(by_dim.values())))
    return by_dim, cas


def all_zero_score_factors(
    factors: Dict[str, Dict[str, float]]
) -> List[str]:
    """Audit-only zero list. Zero is not a separate admission rule in v16.7."""
    zeros = []
    for dim in FACTOR_NAMES:
        for factor, value in factors.get(dim, {}).items():
            if np.isfinite(float(value)) and float(value) <= 0.0:
                zeros.append(f"{dim}.{factor}")
    return zeros


def derive_global_critical_wac(domain: str) -> float:
    """
    Domain Global Critical WAC = mean(minimum critical adequacy rank / 5).

    This is a policy reference used only to resolve the HPS Review band.
    """
    domain = str(domain).lower()
    vals = []
    for dim, factors_ in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factors_:
            vals.append(
                float(DOMAIN_FACTOR_MINIMA[domain][dim][factor])
                / MAX_FACTOR_SCORE
            )
    if not vals:
        raise ValueError(f"No critical factors for domain={domain!r}")
    return float(np.mean(vals))


def critical_factor_audit(
    factors: Dict[str, Dict[str, float]],
    domain: str,
) -> Dict[str, Any]:
    """
    Compute the client Critical WAC_i and the strict individual critical gate.

    Direct Auto-Accept:
        every critical factor >= its own factor-specific adequacy minimum.

    Automated Review:
        uses the overall average Critical WAC_i, allowing compensation across
        critical factors while preserving the dimension and HPS floors.
    """
    domain = str(domain).lower()

    scores = {}
    minima = {}
    missing = []
    below_adequacy = []

    for dim, factor_list in CRITICAL_FACTORS_BY_DOMAIN[domain].items():
        for factor in factor_list:
            key = f"{dim}.{factor}"
            minimum = float(
                DOMAIN_FACTOR_MINIMA[domain][dim][factor]
            )
            minima[key] = minimum

            if factor not in factors.get(dim, {}):
                missing.append(key)
                continue

            value = float(factors[dim][factor])

            if not np.isfinite(value):
                missing.append(key)
                continue

            scores[key] = value

            if value < minimum:
                below_adequacy.append(
                    f"{key}:{value:.1f}<{minimum:.1f}"
                )

    expected_n = sum(
        len(v)
        for v in CRITICAL_FACTORS_BY_DOMAIN[domain].values()
    )

    if missing or len(scores) != expected_n:
        return {
            "scores": scores,
            "minima": minima,
            "missing": missing,
            "critical_count": expected_n,
            "critical_wac_i": np.nan,
            "critical_mean_score_0_5": np.nan,
            "global_critical_wac":
                float(derive_global_critical_wac(domain)),
            "all_critical_meet_adequacy": False,
            "critical_below_adequacy": below_adequacy,
        }

    values = np.array(
        list(scores.values()),
        dtype=float,
    )

    return {
        "scores": scores,
        "minima": minima,
        "missing": [],
        "critical_count": expected_n,
        "critical_wac_i":
            float(np.mean(values / MAX_FACTOR_SCORE)),
        "critical_mean_score_0_5":
            float(np.mean(values)),
        "global_critical_wac":
            float(derive_global_critical_wac(domain)),
        "all_critical_meet_adequacy":
            bool(len(below_adequacy) == 0),
        "critical_below_adequacy":
            below_adequacy,
    }

def dimension_floor_failures(
    dimensions: Dict[str, float],
    floor: float = DIMENSION_MIN_FLOOR,
) -> List[str]:
    """Return averaged HPS dimensions below the minimum floor."""
    return [
        dim for dim, value in dimensions.items()
        if (not np.isfinite(float(value))) or float(value) < float(floor)
    ]

def factor_adequacy_attainment(
    factors: Dict[str, Dict[str, float]],
    factor_minima: Dict[str, Dict[str, float]],
) -> Dict[str, Any]:
    total = 0
    passed = 0
    per_dim = {}

    for dim in FACTOR_NAMES:
        dim_total = 0
        dim_passed = 0

        for factor in FACTOR_NAMES[dim]:
            total += 1
            dim_total += 1

            score = float(factors[dim][factor])
            threshold = float(factor_minima[dim][factor])

            if score >= threshold:
                passed += 1
                dim_passed += 1

        per_dim[dim] = {
            "passed": dim_passed,
            "total": dim_total,
            "fraction": float(dim_passed / max(1, dim_total)),
        }

    return {
        "passed": passed,
        "total": total,
        "fraction": float(passed / max(1, total)),
        "per_dim": per_dim,
    }


def tadp_decision(
    factors: Dict[str, Dict[str, float]],
    domain: str,
    good_cut: float = GOOD_CUT,
    high_cut: float = HIGH_CUT,
) -> Dict[str, Any]:
    """
    v16.7 final fully automated admission policy.

    Flow:
      EVERY dimension >= 2.5?
          NO -> AUTO-REJECT

      HPS >= 3.0?
          NO -> AUTO-REJECT

      HPS >= 3.5?
          YES:
              all critical factors >= own adequacy minima?
                  YES -> DIRECT AUTO-ACCEPT
                  NO  -> AUTOMATED REVIEW
          NO (3.0 <= HPS < 3.5):
              -> AUTOMATED REVIEW

      AUTOMATED REVIEW:
          Critical WAC_i >= Global Critical WAC
              -> ACCEPT AFTER REVIEW
          otherwise
              -> AUTO-REJECT
    """
    domain = str(domain).lower()

    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    dimension_wac = DOMAIN_DIMENSION_WAC[domain]
    global_wac = float(DOMAIN_GLOBAL_WAC[domain])
    global_critical_wac = float(
        derive_global_critical_wac(domain)
    )

    dims = dimension_scores_from_factors(factors)
    hps = compute_hps(dims)

    attainment = factor_adequacy_attainment(
        factors,
        factor_minima,
    )

    zeros = all_zero_score_factors(factors)

    dim_floor_failed = dimension_floor_failures(
        dims,
        floor=DIMENSION_MIN_FLOOR,
    )

    critical = critical_factor_audit(
        factors,
        domain,
    )

    critical_score_string = ";".join(
        (
            f"{key}={value:.1f}"
            f"(min={critical['minima'][key]:.1f})"
        )
        for key, value in critical["scores"].items()
    )

    base = {
        "hps": float(hps),
        "policy_dimension_wac": dimension_wac,
        "global_domain_wac": global_wac,
        "global_critical_wac": global_critical_wac,
        "critical_wac_i": critical["critical_wac_i"],
        "critical_wac_margin": (
            float(critical["critical_wac_i"])
            - global_critical_wac
            if np.isfinite(critical["critical_wac_i"])
            else np.nan
        ),
        "critical_mean_score_0_5":
            critical["critical_mean_score_0_5"],
        "critical_factor_count":
            int(critical["critical_count"]),
        "critical_scores":
            critical_score_string,
        "critical_missing":
            ";".join(critical["missing"]),
        "all_critical_meet_adequacy":
            bool(critical["all_critical_meet_adequacy"]),
        "critical_below_adequacy":
            ";".join(critical["critical_below_adequacy"]),
        "factor_adequacy_passed":
            int(attainment["passed"]),
        "factor_adequacy_total":
            int(attainment["total"]),
        "factor_adequacy_fraction":
            float(attainment["fraction"]),
        "dimensions":
            dims,
        "dimension_min_floor":
            float(DIMENSION_MIN_FLOOR),
        "dimension_floor_failures":
            ";".join(dim_floor_failed),
        # Audit only; zero is not a separate decision rule.
        "all_zero_factors":
            ";".join(zeros),
    }

    # Gate 1: every complete trustworthiness dimension must clear 2.5/5.
    if dim_floor_failed:
        return {
            **base,
            "decision_path":
                "DIMENSION_FLOOR",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_DIMENSION_FLOOR",
            "reason": (
                f"At least one averaged dimension is below "
                f"{DIMENSION_MIN_FLOOR:.1f}/5: "
                + ";".join(
                    f"{dim}={dims[dim]:.3f}"
                    for dim in dim_floor_failed
                )
            ),
        }

    # Missing/non-finite critical evidence is fail-closed in this experiment.
    if critical["missing"]:
        return {
            **base,
            "decision_path":
                "MISSING_CRITICAL_EVIDENCE",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_MISSING_CRITICAL_EVIDENCE",
            "reason": (
                "Missing/non-finite verified critical evidence: "
                + ";".join(critical["missing"])
            ),
        }

    # Gate 2: overall HPS lower bound.
    if hps < float(good_cut):
        return {
            **base,
            "decision_path":
                "LOW_HPS",
            "initial_action":
                "AUTO_REJECT",
            "final_action":
                "REJECT",
            "status":
                "AUTO_REJECTED_LOW_HPS",
            "reason":
                f"HPS {hps:.3f} < {good_cut:.3f}",
        }

    # High-HPS strict direct route.
    if hps >= float(high_cut):
        if critical["all_critical_meet_adequacy"]:
            return {
                **base,
                "decision_path":
                    "DIRECT_AUTO_ACCEPT",
                "initial_action":
                    "AUTO_ACCEPT",
                "final_action":
                    "ACCEPT",
                "status":
                    "DIRECT_AUTO_ACCEPTED",
                "reason": (
                    f"HPS {hps:.3f} >= {high_cut:.3f}; all critical "
                    "factors meet their individual adequacy requirements"
                ),
            }

        # High HPS but failed strict critical gate:
        # fall back to the same automated review rather than immediate rejection.
        review_origin = (
            "HIGH_HPS_CRITICAL_FALLBACK"
        )

    else:
        # 3.0 <= HPS < 3.5
        review_origin = (
            "HPS_REVIEW_BAND"
        )

    # Automated Review for both origins.
    review_pass = bool(
        float(critical["critical_wac_i"])
        >= global_critical_wac
    )

    if review_pass:
        return {
            **base,
            "decision_path":
                review_origin,
            "initial_action":
                "AUTOMATED_REVIEW",
            "final_action":
                "ACCEPT",
            "status":
                "ACCEPTED_AFTER_AUTOMATED_REVIEW",
            "reason": (
                f"{review_origin}: Critical WAC_i "
                f"{critical['critical_wac_i']:.3f} >= "
                f"Global Critical WAC {global_critical_wac:.3f}"
            ),
        }

    return {
        **base,
        "decision_path":
            review_origin,
        "initial_action":
            "AUTOMATED_REVIEW",
        "final_action":
            "REJECT",
        "status":
            "AUTO_REJECTED_REVIEW_CRITICAL_WAC",
        "reason": (
            f"{review_origin}: Critical WAC_i "
            f"{critical['critical_wac_i']:.3f} < "
            f"Global Critical WAC {global_critical_wac:.3f}"
        ),
    }

def build_tadp_governance(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    run: int,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    domain = str(domain).lower()
    rows = []

    for cid in client_ids:
        factors = {
            dim: dict(documentary_evidence[cid][dim])
            for dim in DOCUMENTARY_DIMS
        }

        factors["dim2"] = {
            name: float(dq_scores[cid][name])
            for name in FACTOR_NAMES["dim2"]
        }

        d = tadp_decision(
            factors,
            domain=domain,
        )

        row = {
            "run": int(run),
            "domain": domain,
            "evidence_seed": int(evidence_seed),
            "client": str(cid),
            "hps": float(d["hps"]),
            "global_domain_wac":
                float(d["global_domain_wac"]),
            "global_critical_wac":
                float(d["global_critical_wac"]),
            "critical_wac_i": (
                float(d["critical_wac_i"])
                if np.isfinite(d["critical_wac_i"])
                else np.nan
            ),
            "critical_wac_margin": (
                float(d["critical_wac_margin"])
                if np.isfinite(d["critical_wac_margin"])
                else np.nan
            ),
            "critical_mean_score_0_5": (
                float(d["critical_mean_score_0_5"])
                if np.isfinite(d["critical_mean_score_0_5"])
                else np.nan
            ),
            "critical_factor_count":
                int(d["critical_factor_count"]),
            "critical_scores":
                d["critical_scores"],
            "critical_missing":
                d["critical_missing"],
            "all_critical_meet_adequacy":
                bool(d["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                d["critical_below_adequacy"],
            "dimension_min_floor":
                float(d["dimension_min_floor"]),
            "dimension_floor_failures":
                d["dimension_floor_failures"],
            "factor_adequacy_passed":
                int(d["factor_adequacy_passed"]),
            "factor_adequacy_total":
                int(d["factor_adequacy_total"]),
            "factor_adequacy_fraction":
                float(d["factor_adequacy_fraction"]),
            "decision_path":
                d["decision_path"],
            "initial_action":
                d["initial_action"],
            "final_action":
                d["final_action"],
            "status":
                d["status"],
            "reason":
                d["reason"],
            "all_zero_factors":
                d["all_zero_factors"],
        }

        for dim, value in d["dimensions"].items():
            row[
                f"{dim}_score_0_5"
            ] = float(value)

        for dim, value in d["policy_dimension_wac"].items():
            row[
                f"{dim}_policy_wac"
            ] = float(value)

        rows.append(row)

    return pd.DataFrame(rows)

def accepted_tadp_vr(governance_df: pd.DataFrame) -> List[str]:
    return governance_df.loc[
        governance_df["final_action"].eq("ACCEPT"), "client"
    ].astype(str).tolist()


def accepted_tadp_sda(
    governance_df: pd.DataFrame
) -> List[str]:
    """
    Select one best TADP-eligible client for SDA from already accepted clients.
    """
    eligible = governance_df[
        governance_df["final_action"].eq("ACCEPT")
    ].copy()

    if eligible.empty:
        raise RuntimeError(
            "TADP-SDA cannot select a client: no TADP-eligible client."
        )

    eligible["direct_accept_priority"] = (
        eligible["status"]
        .eq("DIRECT_AUTO_ACCEPTED")
        .astype(int)
    )

    eligible = eligible.sort_values(
        [
            "hps",
            "direct_accept_priority",
            "critical_wac_i",
            "client",
        ],
        ascending=[
            False,
            False,
            False,
            True,
        ],
    )

    return [
        str(
            eligible.iloc[0]["client"]
        )
    ]

def build_full_factor_evidence_table(
    client_ids: List[str],
    documentary_evidence: Dict[str, Dict[str, Dict[str, float]]],
    dq_scores: Dict[str, Dict[str, float]],
    dq_audit: pd.DataFrame,
    evidence_seed: int,
    domain: str,
) -> pd.DataFrame:
    """
    Full 28-factor evidence audit.

    Critical factors are flagged with their own policy adequacy minima.
    Direct Auto-Accept uses these individual minima; Automated Review uses the
    aggregate Critical WAC_i.
    """
    domain = str(domain).lower()
    factor_minima = DOMAIN_FACTOR_MINIMA[domain]
    critical_policy = CRITICAL_FACTORS_BY_DOMAIN[domain]

    dq_index = dq_audit.copy()
    dq_index["client"] = dq_index["client"].astype(str)
    dq_index = dq_index.set_index(
        "client",
        drop=False,
    )

    def policy_fields(dim, factor):
        is_critical = bool(
            factor in critical_policy.get(
                dim,
                [],
            )
        )

        return {
            "is_critical_factor":
                is_critical,
            "critical_adequacy_min_rank": (
                float(factor_minima[dim][factor])
                if is_critical
                else np.nan
            ),
            "critical_adequacy_min_normalized": (
                float(factor_minima[dim][factor])
                / MAX_FACTOR_SCORE
                if is_critical
                else np.nan
            ),
        }

    rows = []

    for cid in client_ids:
        cid = str(cid)

        for dim in DOCUMENTARY_DIMS:
            for factor in FACTOR_NAMES[dim]:
                score = float(
                    documentary_evidence[cid][dim][factor]
                )
                min_rank = float(
                    factor_minima[dim][factor]
                )

                rows.append({
                    "client": cid,
                    "domain": domain,
                    "dimension": dim,
                    "dimension_name":
                        DIMENSION_NAMES[dim],
                    "factor": factor,
                    "factor_source":
                        "CONTROLLED_DOCUMENTARY_EVIDENCE",
                    "raw_measured_value":
                        np.nan,
                    "rubric_score_0_5":
                        score,
                    "rubric_descriptor":
                        rubric_descriptor(
                            dim,
                            factor,
                            score,
                        ),
                    "adequacy_min_rank":
                        min_rank,
                    "adequacy_min_normalized":
                        min_rank / MAX_FACTOR_SCORE,
                    "meets_factor_adequacy":
                        bool(score >= min_rank),
                    **policy_fields(
                        dim,
                        factor,
                    ),
                    "evidence_seed":
                        int(evidence_seed),
                })

        for factor in FACTOR_NAMES["dim2"]:
            score = float(
                dq_scores[cid][factor]
            )
            min_rank = float(
                factor_minima["dim2"][factor]
            )
            raw_col = DQ_RAW_METRIC_BY_FACTOR[
                factor
            ]

            raw_value = (
                float(
                    dq_index.loc[
                        cid,
                        raw_col,
                    ]
                )
                if raw_col in dq_index.columns
                else np.nan
            )

            rows.append({
                "client": cid,
                "domain": domain,
                "dimension": "dim2",
                "dimension_name":
                    DIMENSION_NAMES["dim2"],
                "factor": factor,
                "factor_source":
                    "MACHINE_MEASURED_TRAIN_ONLY",
                "raw_measured_value":
                    raw_value,
                "rubric_score_0_5":
                    score,
                "rubric_descriptor":
                    rubric_descriptor(
                        "dim2",
                        factor,
                        score,
                    ),
                "adequacy_min_rank":
                    min_rank,
                "adequacy_min_normalized":
                    min_rank / MAX_FACTOR_SCORE,
                "meets_factor_adequacy":
                    bool(score >= min_rank),
                **policy_fields(
                    "dim2",
                    factor,
                ),
                "evidence_seed":
                    int(evidence_seed),
            })

    out = pd.DataFrame(
        rows
    )

    expected_rows = len(client_ids) * 28

    if len(out) != expected_rows:
        raise RuntimeError(
            f"Full evidence matrix should contain "
            f"{expected_rows} rows; found {len(out)}."
        )

    return out

# ======================================================================================
# GREAT EXPECTATIONS (GX CORE) MATCHED-K COMPARATOR
# ======================================================================================
# IMPORTANT:
#   Great Expectations is a DATA-VALIDATION framework, not a client-selection algorithm.
#   v16.7 therefore uses REAL GX Core Expectations to validate each client's TRAIN-only
#   data, then applies a separate matched-K ranking policy so the GX comparator has the
#   same client count as TADP-VR. This keeps the downstream training-budget comparison fair.
#
#   The GX validation layer covers:
#       - schema
#       - volume
#       - missingness
#       - uniqueness / duplicates
#       - cardinality / category coverage
#       - value / type consistency
#       - distribution
#       - integrity / constraints
#
#   No TEST data, model metric, HPS, WAC, or TADP decision is used by GX.
GX_CORE_VERSION = "1.23.0"
GX_NONNULL_TOLERANCE = 0.05
GX_CARDINALITY_COVERAGE = 0.60
GX_DISTRIBUTION_SD_TOLERANCE = 0.75
GX_VALUE_MOSTLY = 0.99


def ensure_great_expectations():
    """
    Import GX Core. In Colab, install the pinned version once if unavailable.
    """
    try:
        import great_expectations as gx
        return gx
    except ImportError:
        import subprocess
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                f"great_expectations=={GX_CORE_VERSION}",
            ]
        )
        import great_expectations as gx
        return gx


def _gx_meta(category: str, severity: str, name: str) -> Dict[str, Any]:
    return {
        "dq_category": str(category),
        "policy_severity": str(severity),
        "check_name": str(name),
        "comparator": "GX_CORE_MATCHED_K",
    }


def _gx_add(
    suite,
    specs: List[Dict[str, Any]],
    expectation,
    category: str,
    severity: str,
    name: str,
):
    suite.add_expectation(expectation)
    specs.append({
        "category": str(category),
        "severity": str(severity),
        "name": str(name),
    })


def _aggregate_numeric_reference(
    client_frames: Dict[str, pd.DataFrame],
    numeric_cols: List[str],
) -> Dict[str, Dict[str, float]]:
    """
    TRAIN-only global moments from client-local count/sum/sum-of-squares.
    No TEST data are used.
    """
    out = {}
    for c in numeric_cols:
        count = 0
        total = 0.0
        sumsq = 0.0
        for df in client_frames.values():
            a = pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float)
            a = a[np.isfinite(a)]
            count += int(len(a))
            total += float(a.sum()) if len(a) else 0.0
            sumsq += float(np.square(a).sum()) if len(a) else 0.0

        if count == 0:
            continue

        mean = total / count
        var = max(0.0, sumsq / count - mean * mean)
        out[c] = {
            "count": float(count),
            "mean": float(mean),
            "std": float(math.sqrt(var)),
        }
    return out


def _aggregate_nonnull_reference(
    client_frames: Dict[str, pd.DataFrame],
    columns: List[str],
) -> Dict[str, float]:
    """
    TRAIN-only non-null proportions from client-local counts.
    """
    out = {}
    total_rows = float(sum(len(df) for df in client_frames.values()))

    for c in columns:
        nonnull = 0
        for df in client_frames.values():
            if c in df.columns:
                nonnull += int(df[c].notna().sum())
        out[c] = float(nonnull / max(1.0, total_rows))
    return out


def _aggregate_category_reference(
    client_frames: Dict[str, pd.DataFrame],
    categorical_cols: List[str],
) -> Dict[str, List[str]]:
    """
    TRAIN-only category support as a union of client-local category sets.
    """
    out = {}
    for c in categorical_cols:
        values = set()
        for df in client_frames.values():
            if c in df.columns:
                values.update(
                    df[c]
                    .astype("string")
                    .fillna("__MISSING__")
                    .astype(str)
                    .unique()
                    .tolist()
                )
        out[c] = sorted(values)
    return out


def build_gx_healthcare_reference(
    client_frames: Dict[str, pd.DataFrame],
    preprocessor: Any,
) -> Dict[str, Any]:
    """
    Build a TRAIN-only reference for GX expectations using federated/local summaries.
    """
    first = next(iter(client_frames.values()))
    original_columns = list(first.columns)

    numeric_cols = list(preprocessor.numeric_cols)
    categorical_cols = list(preprocessor.categorical_cols)

    return {
        "original_columns": original_columns,
        "numeric_cols": numeric_cols,
        "categorical_cols": categorical_cols,
        "numeric_reference": _aggregate_numeric_reference(
            client_frames,
            numeric_cols,
        ),
        "nonnull_reference": _aggregate_nonnull_reference(
            client_frames,
            list(preprocessor.feature_cols),
        ),
        "category_reference": _aggregate_category_reference(
            client_frames,
            categorical_cols,
        ),
    }


def build_gx_healthcare_validation_frame(
    df: pd.DataFrame,
    preprocessor: Any,
) -> pd.DataFrame:
    """
    Create a GX validation view of a raw client TRAIN partition.

    Original columns remain present. Numeric candidate columns are converted to
    numeric for GX range/distribution expectations, while explicit diagnostic
    columns preserve type-conversion consistency information.
    """
    out = df.copy()

    for c in preprocessor.numeric_cols:
        raw = df[c]
        parsed = pd.to_numeric(raw, errors="coerce")
        type_ok = raw.isna() | parsed.notna()

        out[c] = parsed.astype(float)
        out[f"__gx_type_ok__{c}"] = type_ok.astype(np.int8)

    return out


def build_gx_healthcare_suite(
    gx,
    reference: Dict[str, Any],
):
    """
    Build a real GX Expectation Suite for structured healthcare TRAIN data.

    Thresholds are fixed comparator settings, not learned from TEST/model outcomes:
      - non-null proportion may be up to 5 percentage points below the TRAIN reference;
      - categorical support must cover >=60% of TRAIN-wide support;
      - numeric means may deviate by +/-0.75 TRAIN-wide standard deviations.
    """
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_healthcare_gx_core_suite")
    specs = []

    original_columns = reference["original_columns"]

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=original_columns,
            exact_match=False,
            severity="critical",
            meta=_gx_meta("Schema", "critical", "required_column_set"),
        ),
        "Schema",
        "critical",
        "required_column_set",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Volume", "warning", "minimum_client_rows"),
        ),
        "Volume",
        "warning",
        "minimum_client_rows",
    )

    if "encounter_id" in original_columns:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeUnique(
                column="encounter_id",
                severity="critical",
                meta=_gx_meta("Uniqueness", "critical", "encounter_id_unique"),
            ),
            "Uniqueness",
            "critical",
            "encounter_id_unique",
        )

    if "_target" in original_columns:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column="_target",
                value_set=[0, 1, 2],
                severity="critical",
                meta=_gx_meta("Integrity", "critical", "target_domain"),
            ),
            "Integrity",
            "critical",
            "target_domain",
        )

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnUniqueValueCountToBeBetween(
                column="_target",
                min_value=3,
                max_value=3,
                severity="warning",
                meta=_gx_meta("Cardinality", "warning", "target_class_cardinality"),
            ),
            "Cardinality",
            "warning",
            "target_class_cardinality",
        )

    # Missingness expectations: compare each feature to TRAIN-only reference.
    for c, global_nonnull in reference["nonnull_reference"].items():
        if c not in original_columns:
            continue
        # Avoid making extremely sparse optional fields dominate the comparator.
        if float(global_nonnull) < 0.10:
            continue

        minimum = max(
            0.0,
            min(
                1.0,
                float(global_nonnull) - GX_NONNULL_TOLERANCE,
            ),
        )

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnProportionOfNonNullValuesToBeBetween(
                column=c,
                min_value=minimum,
                max_value=1.0,
                severity="warning",
                meta=_gx_meta(
                    "Missingness",
                    "warning",
                    f"{c}_nonnull_reference",
                ),
            ),
            "Missingness",
            "warning",
            f"{c}_nonnull_reference",
        )

    # Type/value consistency and numerical distribution.
    for c in reference["numeric_cols"]:
        diag = f"__gx_type_ok__{c}"

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=diag,
                value_set=[1],
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta(
                    "Type Consistency",
                    "critical",
                    f"{c}_numeric_parseability",
                ),
            ),
            "Type Consistency",
            "critical",
            f"{c}_numeric_parseability",
        )

        ref = reference["numeric_reference"].get(c)
        if ref is None:
            continue

        mu = float(ref["mean"])
        sigma = float(ref["std"])

        if np.isfinite(mu) and np.isfinite(sigma) and sigma > 1e-12:
            lo = mu - GX_DISTRIBUTION_SD_TOLERANCE * sigma
            hi = mu + GX_DISTRIBUTION_SD_TOLERANCE * sigma

            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnMeanToBeBetween(
                    column=c,
                    min_value=float(lo),
                    max_value=float(hi),
                    severity="warning",
                    meta=_gx_meta(
                        "Distribution",
                        "warning",
                        f"{c}_mean_reference_band",
                    ),
                ),
                "Distribution",
                "warning",
                f"{c}_mean_reference_band",
            )

    # Domain integrity / range expectations for count-like clinical fields.
    nonnegative_fields = [
        "time_in_hospital",
        "num_lab_procedures",
        "num_procedures",
        "num_medications",
        "number_outpatient",
        "number_emergency",
        "number_inpatient",
        "number_diagnoses",
    ]

    for c in nonnegative_fields:
        if c in original_columns:
            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnValuesToBeBetween(
                    column=c,
                    min_value=0.0,
                    mostly=GX_VALUE_MOSTLY,
                    severity="critical",
                    meta=_gx_meta(
                        "Integrity",
                        "critical",
                        f"{c}_nonnegative",
                    ),
                ),
                "Integrity",
                "critical",
                f"{c}_nonnegative",
            )

    # Cardinality and allowed-support consistency for categorical features.
    for c, global_values in reference["category_reference"].items():
        if c not in original_columns or not global_values:
            continue

        g = int(len(global_values))

        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c,
                value_set=list(global_values),
                mostly=GX_VALUE_MOSTLY,
                severity="critical",
                meta=_gx_meta(
                    "Value Consistency",
                    "critical",
                    f"{c}_allowed_train_support",
                ),
            ),
            "Value Consistency",
            "critical",
            f"{c}_allowed_train_support",
        )

        if g >= 2:
            min_unique = max(
                1,
                int(math.ceil(GX_CARDINALITY_COVERAGE * g)),
            )

            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnUniqueValueCountToBeBetween(
                    column=c,
                    min_value=min_unique,
                    max_value=g,
                    severity="warning",
                    meta=_gx_meta(
                        "Cardinality",
                        "warning",
                        f"{c}_support_coverage",
                    ),
                ),
                "Cardinality",
                "warning",
                f"{c}_support_coverage",
            )

    return suite, specs


def build_gx_cifar_metadata(
    X: np.ndarray,
    y: np.ndarray,
) -> pd.DataFrame:
    """
    Structured per-image metadata for GX validation of CIFAR-10.
    GX officially supports validating structured metadata derived from
    unstructured data; raw pixels remain client-local.
    """
    X = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    if X.ndim != 4:
        raise RuntimeError(
            f"Expected CIFAR tensor [N,H,W,C], got shape={X.shape}"
        )

    finite = np.isfinite(
        X.astype(np.float32)
    ).reshape(len(X), -1).all(axis=1)

    return pd.DataFrame({
        "label": y.astype(np.int32),
        "height": np.full(len(X), X.shape[1], dtype=np.int32),
        "width": np.full(len(X), X.shape[2], dtype=np.int32),
        "channels": np.full(len(X), X.shape[3], dtype=np.int32),
        "dtype_ok": np.full(
            len(X),
            int(X.dtype == np.uint8),
            dtype=np.int8,
        ),
        "finite": finite.astype(np.int8),
        "pixel_min": X.reshape(len(X), -1).min(axis=1).astype(float),
        "pixel_max": X.reshape(len(X), -1).max(axis=1).astype(float),
        "pixel_mean": X.reshape(len(X), -1).mean(axis=1).astype(float),
        "pixel_std": X.reshape(len(X), -1).std(axis=1).astype(float),
    })


def build_gx_cifar_reference(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]],
) -> Dict[str, Any]:
    """
    TRAIN-only image-metadata reference from client-local summary statistics.
    """
    total_n = 0
    sum_mean = 0.0
    sumsq_mean = 0.0
    sum_std = 0.0
    sumsq_std = 0.0

    for X, y in client_raw.values():
        md = build_gx_cifar_metadata(X, y)

        for col in ["pixel_mean", "pixel_std"]:
            a = md[col].to_numpy(dtype=float)
            if col == "pixel_mean":
                sum_mean += float(a.sum())
                sumsq_mean += float(np.square(a).sum())
            else:
                sum_std += float(a.sum())
                sumsq_std += float(np.square(a).sum())

        total_n += len(md)

    n = max(1, int(total_n))

    mean_mu = sum_mean / n
    mean_var = max(0.0, sumsq_mean / n - mean_mu * mean_mu)

    std_mu = sum_std / n
    std_var = max(0.0, sumsq_std / n - std_mu * std_mu)

    return {
        "pixel_mean": {
            "mean": float(mean_mu),
            "std": float(math.sqrt(mean_var)),
        },
        "pixel_std": {
            "mean": float(std_mu),
            "std": float(math.sqrt(std_var)),
        },
    }


def build_gx_cifar_suite(
    gx,
    reference: Dict[str, Any],
):
    gxe = gx.expectations
    suite = gx.ExpectationSuite(name="tadp_cifar10_gx_core_suite")
    specs = []

    columns = [
        "label",
        "height",
        "width",
        "channels",
        "dtype_ok",
        "finite",
        "pixel_min",
        "pixel_max",
        "pixel_mean",
        "pixel_std",
    ]

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableColumnsToMatchSet(
            column_set=columns,
            exact_match=True,
            severity="critical",
            meta=_gx_meta("Schema", "critical", "image_metadata_schema"),
        ),
        "Schema",
        "critical",
        "image_metadata_schema",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectTableRowCountToBeBetween(
            min_value=100,
            severity="warning",
            meta=_gx_meta("Volume", "warning", "minimum_client_images"),
        ),
        "Volume",
        "warning",
        "minimum_client_images",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectColumnValuesToBeInSet(
            column="label",
            value_set=list(range(10)),
            severity="critical",
            meta=_gx_meta("Integrity", "critical", "label_domain"),
        ),
        "Integrity",
        "critical",
        "label_domain",
    )

    _gx_add(
        suite,
        specs,
        gxe.ExpectColumnUniqueValueCountToBeBetween(
            column="label",
            min_value=8,
            max_value=10,
            severity="warning",
            meta=_gx_meta("Cardinality", "warning", "label_support"),
        ),
        "Cardinality",
        "warning",
        "label_support",
    )

    for c, value in [
        ("height", 32),
        ("width", 32),
        ("channels", 3),
        ("dtype_ok", 1),
        ("finite", 1),
    ]:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeInSet(
                column=c,
                value_set=[value],
                severity="critical",
                meta=_gx_meta(
                    "Integrity",
                    "critical",
                    f"{c}_constraint",
                ),
            ),
            "Integrity",
            "critical",
            f"{c}_constraint",
        )

    for c in ["pixel_min", "pixel_max"]:
        _gx_add(
            suite,
            specs,
            gxe.ExpectColumnValuesToBeBetween(
                column=c,
                min_value=0.0,
                max_value=255.0,
                severity="critical",
                meta=_gx_meta(
                    "Value Consistency",
                    "critical",
                    f"{c}_valid_range",
                ),
            ),
            "Value Consistency",
            "critical",
            f"{c}_valid_range",
        )

    for c in ["pixel_mean", "pixel_std"]:
        mu = float(reference[c]["mean"])
        sigma = float(reference[c]["std"])

        if sigma > 1e-12:
            lo = mu - GX_DISTRIBUTION_SD_TOLERANCE * sigma
            hi = mu + GX_DISTRIBUTION_SD_TOLERANCE * sigma

            _gx_add(
                suite,
                specs,
                gxe.ExpectColumnMeanToBeBetween(
                    column=c,
                    min_value=float(lo),
                    max_value=float(hi),
                    severity="warning",
                    meta=_gx_meta(
                        "Distribution",
                        "warning",
                        f"{c}_mean_reference_band",
                    ),
                ),
                "Distribution",
                "warning",
                f"{c}_mean_reference_band",
            )

    return suite, specs


def _gx_result_unexpected_percent(result_obj) -> float:
    try:
        payload = result_obj.result
        if payload is None:
            return float("nan")
        if hasattr(payload, "get"):
            value = payload.get("unexpected_percent", np.nan)
        else:
            value = np.nan
        return float(value) if value is not None else float("nan")
    except Exception:
        return float("nan")


def ge_governance(
    client_data: Dict[str, Any],
    client_ids: List[str],
    accept_count: int,
    domain: str,
    preprocessor: Optional[Any] = None,
    dq_scores: Optional[Dict[str, Dict[str, float]]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    REAL Great Expectations GX Core comparator + matched-K selection.

    GX performs TRAIN-only validation. GX itself does NOT define "select exactly
    K clients." After validation, this experiment ranks clients by GX validation
    quality and selects the top K, where K equals the frozen TADP-VR cohort size.

    Matched-K ranking:
      1) fewer critical GX failures;
      2) higher GX expectation pass rate;
      3) fewer warning failures;
      4) lower mean unexpected percentage across row-level GX checks;
      5) deterministic client ID tie-break.

    All clients remain rankable. Exactly K are selected for downstream training,
    where K is the frozen TADP-VR cohort size. This is an experimental matched-K
    control layered on top of GX validation, not a native GX admission rule.
    """
    domain = str(domain).lower()
    k = min(int(accept_count), len(client_ids))

    if k <= 0:
        raise RuntimeError("GX matched-K comparator requires K >= 1.")

    gx = ensure_great_expectations()

    context = gx.get_context(mode="ephemeral")
    datasource = context.data_sources.add_pandas(
        name=f"tadp_gx_{domain}_datasource"
    )
    asset = datasource.add_dataframe_asset(
        name=f"tadp_gx_{domain}_client_asset"
    )
    batch_definition = asset.add_batch_definition_whole_dataframe(
        name="whole_client_train_partition"
    )

    if domain == "healthcare":
        if preprocessor is None:
            raise ValueError(
                "Healthcare GX comparator requires the TRAIN-only preprocessor."
            )
        reference = build_gx_healthcare_reference(
            client_data,
            preprocessor,
        )
        suite, specs = build_gx_healthcare_suite(
            gx,
            reference,
        )

        def make_frame(cid):
            return build_gx_healthcare_validation_frame(
                client_data[cid],
                preprocessor,
            )

    elif domain == "cifar10":
        reference = build_gx_cifar_reference(
            client_data
        )
        suite, specs = build_gx_cifar_suite(
            gx,
            reference,
        )

        def make_frame(cid):
            X, y = client_data[cid]
            return build_gx_cifar_metadata(
                X,
                y,
            )

    else:
        raise ValueError(
            f"Unsupported GX domain: {domain!r}"
        )

    summary_rows = []
    detail_rows = []

    for cid in client_ids:
        frame = make_frame(cid)

        batch = batch_definition.get_batch(
            batch_parameters={
                "dataframe": frame
            }
        )

        validation = batch.validate(
            suite
        )

        results = list(
            validation.results
        )

        if len(results) != len(specs):
            raise RuntimeError(
                f"GX result/spec mismatch for client {cid}: "
                f"{len(results)} results vs {len(specs)} expectations."
            )

        passed = 0
        critical_failures = 0
        warning_failures = 0
        unexpected_values = []

        category_totals = {}
        category_passed = {}

        for spec, result in zip(
            specs,
            results,
        ):
            success = bool(
                result.success
            )

            passed += int(success)

            category = spec["category"]
            severity = spec["severity"]

            category_totals[category] = (
                category_totals.get(
                    category,
                    0,
                ) + 1
            )
            category_passed[category] = (
                category_passed.get(
                    category,
                    0,
                ) + int(success)
            )

            if not success:
                if severity == "critical":
                    critical_failures += 1
                else:
                    warning_failures += 1

            unexpected_percent = (
                _gx_result_unexpected_percent(
                    result
                )
            )

            if np.isfinite(
                unexpected_percent
            ):
                unexpected_values.append(
                    unexpected_percent
                )

            detail_rows.append({
                "client":
                    str(cid),
                "domain":
                    domain,
                "gx_version":
                    GX_CORE_VERSION,
                "expectation_name":
                    spec["name"],
                "dq_category":
                    category,
                "severity":
                    severity,
                "success":
                    success,
                "unexpected_percent":
                    unexpected_percent,
            })

        total = len(specs)
        pass_rate = (
            passed / max(1, total)
        )

        mean_unexpected = (
            float(np.mean(unexpected_values))
            if unexpected_values
            else 0.0
        )

        row = {
            "client":
                str(cid),
            "gx_version":
                GX_CORE_VERSION,
            "gx_native_suite_success":
                bool(validation.success),
            "ge_expectations_passed":
                int(passed),
            "ge_expectations_total":
                int(total),
            "ge_pass_rate":
                float(pass_rate),
            "ge_critical_failures":
                int(critical_failures),
            "ge_warning_failures":
                int(warning_failures),
            "ge_mean_unexpected_percent":
                float(mean_unexpected),
        }

        for category in sorted(
            category_totals
        ):
            safe = re.sub(
                r"[^a-z0-9]+",
                "_",
                category.lower(),
            ).strip("_")

            row[
                f"ge_{safe}_passed"
            ] = int(
                category_passed.get(
                    category,
                    0,
                )
            )
            row[
                f"ge_{safe}_total"
            ] = int(
                category_totals[
                    category
                ]
            )

        if dq_scores is not None:
            vals = np.array(
                [
                    float(
                        dq_scores[cid][f]
                    )
                    for f in FACTOR_NAMES["dim2"]
                ],
                dtype=float,
            )
            row["ge_descriptive_mean_dq_score"] = float(
                np.mean(vals)
            )

        summary_rows.append(
            row
        )

    summary = pd.DataFrame(
        summary_rows
    )

    # ------------------------------------------------------------------
    # Matched-K comparator selection.
    #
    # GX validates data; it does not define a native "select exactly K"
    # admission rule. For a fair downstream comparison with TADP-VR, ALL
    # clients are ranked by observed GX validation quality and exactly the
    # top K are selected.
    #
    # v16.7 incorrectly used zero critical failures as a hard eligibility
    # filter. If all clients fail at least one strict critical Expectation,
    # that prevents matched-K comparison entirely. v16.7 keeps critical
    # failures as the STRONGEST ranking criterion, but not as a hard filter.
    #
    # Lexicographic ranking (best first):
    #   1) fewer critical GX failures
    #   2) higher overall GX Expectation pass rate
    #   3) fewer warning failures
    #   4) lower mean unexpected percentage
    #   5) deterministic client-ID tie-break
    # ------------------------------------------------------------------
    summary = summary.sort_values(
        [
            "ge_critical_failures",
            "ge_pass_rate",
            "ge_warning_failures",
            "ge_mean_unexpected_percent",
            "client",
        ],
        ascending=[
            True,
            False,
            True,
            True,
            True,
        ],
    ).reset_index(drop=True)

    summary["ge_rank"] = np.arange(1, len(summary) + 1)

    selected = set(
        summary.head(k)["client"].astype(str)
    )

    summary["ge_matched_k_selected"] = (
        summary["client"].astype(str).isin(selected)
    )

    summary["ge_final_action"] = (
        summary["ge_matched_k_selected"].map({
            True: "ACCEPT",
            False: "REJECT",
        })
    )

    summary["ge_native_validation_class"] = np.select(
        [
            summary["ge_critical_failures"].eq(0)
            & summary["ge_warning_failures"].eq(0),
            summary["ge_critical_failures"].eq(0),
        ],
        [
            "GX_ALL_EXPECTATIONS_PASSED",
            "GX_WARNING_FAILURES_ONLY",
        ],
        default="GX_HAS_CRITICAL_FAILURES",
    )

    summary["ge_policy"] = (
        f"GX Core {GX_CORE_VERSION} TRAIN-only validation + "
        f"matched-K lexicographic top-{k}/{len(summary)} selection; "
        "ranking prioritizes fewer critical failures, higher pass rate, "
        "fewer warning failures, then lower unexpected percentage"
    )

    details = pd.DataFrame(
        detail_rows
    ).sort_values(
        [
            "client",
            "dq_category",
            "expectation_name",
        ]
    ).reset_index(
        drop=True
    )

    return summary, details

def score_lower_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [best_upper, score4_upper, score3_upper, score2_upper, score1_upper]
    value <= cuts[0] => 5; ... value <= cuts[4] => 1; else 0.
    """
    v = float(value)
    for score, upper in zip([5, 4, 3, 2, 1], cuts):
        if v <= float(upper):
            return float(score)
    return 0.0


def score_higher_is_better(value: float, cuts: List[float]) -> float:
    """
    cuts = [score5_lower, score4_lower, score3_lower, score2_lower, score1_lower]
    """
    v = float(value)
    for score, lower in zip([5, 4, 3, 2, 1], cuts):
        if v >= float(lower):
            return float(score)
    return 0.0


def js_divergence(p, q, eps=1e-12) -> float:
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    p = p / max(eps, p.sum())
    q = q / max(eps, q.sum())
    m = 0.5 * (p + q)
    kl_pm = np.sum(np.where(p > 0, p * np.log((p + eps) / (m + eps)), 0.0))
    kl_qm = np.sum(np.where(q > 0, q * np.log((q + eps) / (m + eps)), 0.0))
    return float(0.5 * (kl_pm + kl_qm))


# ======================================================================================
# MODEL / METRICS / FL TRAINING
# ======================================================================================

def class_weight_dict(y: np.ndarray) -> Dict[int, float]:
    y = np.asarray(y, dtype=np.int32)
    classes = np.unique(y)
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return {int(c): float(w) for c, w in zip(classes, weights)}


def model_parameter_bytes(model: keras.Model) -> int:
    return int(sum(np.asarray(w).nbytes for w in model.get_weights()))


def evaluate_model(
    model: keras.Model, X: np.ndarray, y: np.ndarray, n_classes: int
) -> Dict[str, float]:
    p = model.predict(X, batch_size=512, verbose=0)
    pred = np.argmax(p, axis=1)
    out = {
        "accuracy": float(accuracy_score(y, pred)),
        "precision_macro": float(
            precision_score(y, pred, average="macro", zero_division=0)
        ),
        "recall_macro": float(
            recall_score(y, pred, average="macro", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y, pred, average="macro", zero_division=0)
        ),
    }
    try:
        out["roc_auc_ovr_macro"] = float(
            roc_auc_score(y, p, multi_class="ovr", average="macro")
        )
    except Exception:
        out["roc_auc_ovr_macro"] = float("nan")
    return out


def train_exact_steps(
    model: keras.Model,
    X: np.ndarray,
    y: np.ndarray,
    steps: int,
    batch_size: int,
    seed: int,
    class_weights: Optional[Dict[int, float]] = None,
    prox_reference: Optional[List[np.ndarray]] = None,
    prox_mu: float = 0.0,
):
    """
    Exact mini-batch update count. Used for step-parity audits.
    """
    steps = int(max(1, steps))
    rng = np.random.default_rng(int(seed))
    n = len(y)
    if n == 0:
        raise RuntimeError("Cannot train on an empty dataset.")

    loss_fn = keras.losses.SparseCategoricalCrossentropy(
        reduction=keras.losses.Reduction.NONE
    )

    order = rng.permutation(n)
    cursor = 0

    prox_tensors = None
    if prox_reference is not None and prox_mu > 0:
        prox_tensors = [tf.convert_to_tensor(w) for w in prox_reference]

    for _ in range(steps):
        if cursor + batch_size > n:
            order = rng.permutation(n)
            cursor = 0

        idx = order[cursor:cursor + batch_size]
        cursor += batch_size

        xb = tf.convert_to_tensor(np.asarray(X[idx]), dtype=tf.float32)
        yb_np = np.asarray(y[idx], dtype=np.int32)
        yb = tf.convert_to_tensor(yb_np, dtype=tf.int32)

        with tf.GradientTape() as tape:
            probs = model(xb, training=True)
            per_loss = loss_fn(yb, probs)

            if class_weights:
                sw = np.array(
                    [class_weights.get(int(v), 1.0) for v in yb_np],
                    dtype=np.float32,
                )
                sw_t = tf.convert_to_tensor(sw)
                data_loss = tf.reduce_sum(per_loss * sw_t) / tf.reduce_sum(sw_t)
            else:
                data_loss = tf.reduce_mean(per_loss)

            loss = data_loss

            if prox_tensors is not None:
                prox = tf.constant(0.0, dtype=tf.float32)
                for var, ref in zip(model.trainable_variables, prox_tensors):
                    prox += tf.reduce_sum(tf.square(var - tf.cast(ref, var.dtype)))
                loss = loss + 0.5 * float(prox_mu) * prox

        grads = tape.gradient(loss, model.trainable_variables)
        model.optimizer.apply_gradients(zip(grads, model.trainable_variables))


def natural_steps(n_records: int, batch_size: int, local_epochs: int = 1) -> int:
    return int(max(1, math.ceil(int(n_records) / int(batch_size)) * int(local_epochs)))


def allocate_exact_step_budget(
    selected: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    target_total: int,
    batch_size: int,
    local_epochs: int = 1,
) -> Dict[str, int]:
    natural = {
        cid: natural_steps(len(client_arrays[cid][1]), batch_size, local_epochs)
        for cid in selected
    }
    total_nat = max(1, sum(natural.values()))
    raw = {cid: target_total * natural[cid] / total_nat for cid in selected}
    alloc = {cid: max(1, int(math.floor(raw[cid]))) for cid in selected}

    # Adjust to exact target.
    while sum(alloc.values()) < target_total:
        cid = max(selected, key=lambda c: raw[c] - alloc[c])
        alloc[cid] += 1
    while sum(alloc.values()) > target_total:
        candidates = [c for c in selected if alloc[c] > 1]
        if not candidates:
            break
        cid = min(candidates, key=lambda c: raw[c] - alloc[c])
        alloc[cid] -= 1

    if sum(alloc.values()) != int(target_total):
        raise RuntimeError("Exact step-budget allocation failed.")
    return alloc


def aggregate_weights(
    local_weights: List[List[np.ndarray]],
    sample_sizes: List[int],
    equal_weight: bool = False,
) -> List[np.ndarray]:
    if not local_weights:
        raise RuntimeError("No local weights to aggregate.")
    if equal_weight:
        alpha = np.ones(len(local_weights), dtype=float) / len(local_weights)
    else:
        sizes = np.asarray(sample_sizes, dtype=float)
        alpha = sizes / sizes.sum()

    out = []
    for layer_idx in range(len(local_weights[0])):
        x = sum(alpha[j] * np.asarray(local_weights[j][layer_idx])
                for j in range(len(local_weights)))
        out.append(np.asarray(x))
    return out


def federated_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_per_round: List[List[str]],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    local_epochs: int,
    class_weights: Dict[int, float],
    run_seed: int,
    exact_step_maps: Optional[List[Dict[str, int]]] = None,
    equal_weight: bool = False,
    fedprox_mu: float = 0.0,
) -> Dict[str, Any]:
    monitor = RAMMonitor().start()
    start = time.perf_counter()

    global_model = build_model_fn()
    global_model.set_weights([np.array(w, copy=True) for w in initial_weights])

    round_rows = []
    total_steps = 0
    total_selected = 0

    for r, selected in enumerate(selected_per_round, start=1):
        global_weights = [np.array(w, copy=True) for w in global_model.get_weights()]
        local_weights = []
        local_sizes = []
        round_steps = 0

        if exact_step_maps is None:
            step_map = {
                cid: natural_steps(
                    len(client_arrays[cid][1]), batch_size, local_epochs
                )
                for cid in selected
            }
        else:
            step_map = exact_step_maps[r - 1]

        for j, cid in enumerate(selected):
            Xc, yc = client_arrays[cid]
            local_model = build_model_fn()
            local_model.set_weights(global_weights)
            train_exact_steps(
                local_model, Xc, yc,
                steps=int(step_map[cid]),
                batch_size=batch_size,
                seed=int(run_seed + 1000 * r + 17 * j),
                class_weights=class_weights,
                prox_reference=global_weights if fedprox_mu > 0 else None,
                prox_mu=float(fedprox_mu),
            )
            local_weights.append(
                [np.array(w, copy=True) for w in local_model.get_weights()]
            )
            local_sizes.append(len(yc))
            round_steps += int(step_map[cid])

            del local_model
            tf.keras.backend.clear_session()

        agg = aggregate_weights(
            local_weights, local_sizes, equal_weight=equal_weight
        )
        global_model = build_model_fn()
        global_model.set_weights(agg)

        total_steps += round_steps
        total_selected += len(selected)
        round_rows.append({
            "round": r,
            "selected_clients": len(selected),
            "selected_ids": ";".join(selected),
            "optimizer_steps": int(round_steps),
        })
        print(
            f"      ✅ ROUND {r}/{len(selected_per_round)} DONE | "
            f"clients={len(selected)} | optimizer_steps={round_steps} | "
            f"cumulative_steps={total_steps}"
        )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(global_model, X_test, y_test, n_classes)
    param_b = model_parameter_bytes(global_model)
    ram = monitor.stop()

    # Two-way model traffic (download + upload), plus 12% protocol overhead.
    communication_b = 0
    for row in round_rows:
        communication_b += int(row["selected_clients"]) * 2 * param_b
    communication_b = int(communication_b * 1.12)

    return {
        "model": global_model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(total_steps),
        "communication_mb": float(communication_b / (1024 ** 2)),
        "round_audit": round_rows,
        "participants_mean_per_round": float(
            total_selected / max(1, len(round_rows))
        ),
        **ram,
    }


def centralized_train(
    build_model_fn,
    initial_weights: List[np.ndarray],
    selected_clients: List[str],
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]],
    X_test: np.ndarray,
    y_test: np.ndarray,
    n_classes: int,
    batch_size: int,
    exact_steps: int,
    class_weights: Dict[int, float],
    seed: int,
) -> Dict[str, Any]:
    if not selected_clients:
        raise RuntimeError("Centralized scenario has no TRAIN clients.")

    monitor = RAMMonitor().start()
    start = time.perf_counter()

    # Pool ACCEPTED TRAIN partitions only. TEST is not present here.
    X = np.concatenate([client_arrays[c][0] for c in selected_clients], axis=0)
    y = np.concatenate([client_arrays[c][1] for c in selected_clients], axis=0)

    model = build_model_fn()
    model.set_weights([np.array(w, copy=True) for w in initial_weights])
    train_exact_steps(
        model, X, y,
        steps=int(exact_steps),
        batch_size=batch_size,
        seed=int(seed),
        class_weights=class_weights,
    )

    runtime = time.perf_counter() - start
    metrics = evaluate_model(model, X_test, y_test, n_classes)
    ram = monitor.stop()

    return {
        "model": model,
        "metrics": metrics,
        "runtime_s": float(runtime),
        "total_optimizer_steps": int(exact_steps),
        "communication_mb": 0.0,
        "participants_mean_per_round": float(len(selected_clients)),
        **ram,
    }


def result_row(
    run: int,
    seed: int,
    scenario: str,
    result: Dict[str, Any],
    initial_hash: str,
    power_w: float = 12.0,
) -> Dict[str, Any]:
    row = {
        "run": int(run),
        "seed": int(seed),
        "scenario": str(scenario),
        **result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "participants": float(result.get("participants_mean_per_round", 0.0)),
        "ram_start_mb": float(result.get("ram_start_mb", 0.0)),
        "ram_end_mb": float(result.get("ram_end_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
        "ram_delta_mb": float(result.get("ram_delta_mb", 0.0)),
        "ram_mb": float(result.get("ram_mb", result.get("ram_peak_mb", 0.0))),
        "initial_weights_sha256": initial_hash,
    }
    row["energy_wh"] = float(power_w * row["runtime_s"] / 3600.0)
    row["energy_kwh"] = float(row["energy_wh"] / 1000.0)
    row["co2_kg"] = float(row["energy_kwh"] * 0.430)
    row["energy_cost_usd"] = float(row["energy_kwh"] * 0.20)
    row["communication_cost_usd"] = float(row["communication_mb"] * 0.005)
    row["total_estimated_cost_usd"] = float(
        row["energy_cost_usd"] + row["communication_cost_usd"]
    )
    return row


# ======================================================================================
# LEAKAGE AUDIT
# ======================================================================================

def write_leakage_audit(
    out_dir: Path,
    train_ids,
    test_ids,
    client_train_ids: Dict[str, np.ndarray],
    extra: Optional[Dict[str, Any]] = None,
):
    train_set = set(map(str, train_ids))
    test_set = set(map(str, test_ids))
    overlap = train_set & test_set

    client_union = set()
    duplicates_across_clients = 0
    for cid, ids in client_train_ids.items():
        s = set(map(str, ids))
        duplicates_across_clients += len(client_union & s)
        client_union |= s

    test_in_clients = len(test_set & client_union)
    missing_train = len(train_set - client_union)
    extra_client_rows = len(client_union - train_set)

    row = {
        "train_test_overlap": len(overlap),
        "test_rows_in_any_client": test_in_clients,
        "train_rows_missing_from_clients": missing_train,
        "client_rows_not_in_global_train": extra_client_rows,
        "duplicate_train_rows_across_clients": duplicates_across_clients,
        "pass": (
            len(overlap) == 0
            and test_in_clients == 0
            and missing_train == 0
            and extra_client_rows == 0
            and duplicates_across_clients == 0
        ),
    }
    if extra:
        row.update(extra)

    pd.DataFrame([row]).to_csv(
        Path(out_dir) / "leakage_audit.csv", index=False
    )

    if not bool(row["pass"]):
        raise RuntimeError(f"FAIL-CLOSED leakage audit failed: {row}")
    return row



# ======================================================================================
# CIFAR-10 — LEAKAGE-SAFE DATA PREPARATION
# ======================================================================================

def dirichlet_partition_indices(
    y: np.ndarray,
    n_clients: int,
    alpha: float,
    seed: int,
    min_client_records: int = 100,
) -> Dict[str, np.ndarray]:
    y = np.asarray(y).reshape(-1)
    client_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")[:n_clients]

    for attempt in range(100):
        rng = np.random.default_rng(int(seed + attempt))
        buckets = [[] for _ in range(n_clients)]
        for cls in sorted(np.unique(y)):
            idx = np.where(y == cls)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(np.full(n_clients, float(alpha)))
            cuts = (np.cumsum(props)[:-1] * len(idx)).astype(int)
            splits = np.split(idx, cuts)
            for k, s in enumerate(splits):
                buckets[k].extend(s.tolist())

        if min(len(b) for b in buckets) >= int(min_client_records):
            return {
                cid: np.asarray(b, dtype=np.int64)
                for cid, b in zip(client_ids, buckets)
            }

    raise RuntimeError("Could not obtain valid CIFAR Dirichlet partition.")


def cifar_train_only_mean_std(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> Tuple[np.ndarray, np.ndarray]:
    count = 0
    sum_ = np.zeros(3, dtype=np.float64)
    sumsq = np.zeros(3, dtype=np.float64)

    for X, _ in client_raw.values():
        a = X.astype(np.float64) / 255.0
        count += int(np.prod(a.shape[:3]))
        sum_ += a.sum(axis=(0, 1, 2))
        sumsq += np.square(a).sum(axis=(0, 1, 2))

    mean = sum_ / max(1, count)
    var = np.maximum(1e-12, sumsq / max(1, count) - np.square(mean))
    std = np.sqrt(var)
    return mean.astype(np.float32), std.astype(np.float32)


def cifar_reference_histograms(
    client_raw: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> Dict[str, Any]:
    hist = np.zeros((3, 16), dtype=np.float64)
    class_counts = np.zeros(10, dtype=np.float64)

    for X, y in client_raw.values():
        x = X.astype(np.float32) / 255.0
        for ch in range(3):
            h, _ = np.histogram(x[..., ch], bins=np.linspace(0, 1, 17))
            hist[ch] += h
        class_counts += np.bincount(
            np.asarray(y).reshape(-1), minlength=10
        )

    return {"hist": hist, "class_counts": class_counts}


def cifar_dq_scores(
    X: np.ndarray,
    y: np.ndarray,
    reference: Dict[str, Any],
) -> Tuple[Dict[str, float], Dict[str, float]]:
    """
    Eight machine-measured TRAIN-only DQ factors for CIFAR-10.
    """
    x = np.asarray(X)
    y = np.asarray(y).reshape(-1)

    # 1) Completeness.
    missing_fraction = float(
        np.mean(~np.isfinite(x.astype(np.float32)))
    )
    completeness = score_lower_is_better(
        missing_fraction,
        [0.01, 0.05, 0.10, 0.20, 0.50],
    )

    # 2) Duplication rate — deterministic sample hash audit.
    sample_n = min(len(x), 2000)
    hashes = [
        hashlib.sha1(
            np.ascontiguousarray(x[i]).tobytes()
        ).digest()
        for i in range(sample_n)
    ]
    duplicate_fraction = float(
        1.0 - len(set(hashes)) / max(1, len(hashes))
    )
    duplication = score_lower_is_better(
        duplicate_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 3) Value validity / error rate.
    xf = x.astype(np.float32)
    invalid_value = (
        (~np.isfinite(xf))
        | (xf < 0)
        | (xf > 255)
    )
    error_fraction = float(np.mean(invalid_value))
    value_validity_error_rate = score_lower_is_better(
        error_fraction,
        [0.01, 0.02, 0.05, 0.10, 0.15],
    )

    # 4) Type consistency.
    type_inconsistency = 0.0 if x.dtype == np.uint8 else 1.0
    type_consistency = score_lower_is_better(
        type_inconsistency,
        [0.01, 0.02, 0.05, 0.10, 0.20],
    )

    # 5) Label integrity.
    invalid_label_fraction = float(
        np.mean((y < 0) | (y > 9))
    )
    label_integrity = score_lower_is_better(
        invalid_label_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    # 6) RGB distribution consistency against TRAIN-only aggregate.
    xn = xf / 255.0
    jsds = []

    for ch in range(3):
        h, _ = np.histogram(
            xn[..., ch],
            bins=np.linspace(0, 1, 17),
        )
        jsds.append(
            js_divergence(
                h,
                reference["hist"][ch],
            )
        )

    max_jsd = float(max(jsds))
    distribution_consistency = score_lower_is_better(
        max_jsd,
        [0.01, 0.025, 0.05, 0.10, 0.20],
    )

    # 7) Class/category coverage.
    present = int(
        np.sum(
            np.bincount(
                y,
                minlength=10,
            ) > 0
        )
    )
    mean_coverage = float(present / 10.0)
    feature_coverage = score_higher_is_better(
        mean_coverage,
        [0.90, 0.825, 0.75, 0.65, 0.50],
    )

    # 8) Structural / constraint integrity.
    # CIFAR-specific structural expectations:
    #   rank-4 batch, 32x32x3 samples, uint8 raw pixels,
    #   finite values, valid range, valid labels.
    if x.ndim != 4 or x.shape[1:] != (32, 32, 3):
        structural_violation_fraction = 1.0
    else:
        sample_bad = np.zeros(len(x), dtype=bool)

        # Any nonfinite/out-of-range pixel makes that sample structurally invalid.
        pixel_bad = (
            (~np.isfinite(xf))
            | (xf < 0)
            | (xf > 255)
        )
        sample_bad |= np.any(
            pixel_bad.reshape(len(x), -1),
            axis=1,
        )

        if x.dtype != np.uint8:
            sample_bad[:] = True

        if len(y) != len(x):
            structural_violation_fraction = 1.0
        else:
            sample_bad |= ((y < 0) | (y > 9))
            structural_violation_fraction = float(
                np.mean(sample_bad)
            )

    structural_integrity = score_lower_is_better(
        structural_violation_fraction,
        [0.001, 0.01, 0.02, 0.05, 0.10],
    )

    scores = {
        "completeness": completeness,
        "duplication_rate": duplication,
        "value_validity_error_rate": value_validity_error_rate,
        "type_consistency": type_consistency,
        "label_integrity": label_integrity,
        "feature_distribution_consistency": distribution_consistency,
        "feature_category_coverage": feature_coverage,
        "structural_constraint_integrity": structural_integrity,
    }

    raw = {
        "missing_fraction": missing_fraction,
        "duplicate_fraction": duplicate_fraction,
        "error_fraction": error_fraction,
        "type_inconsistency_fraction": type_inconsistency,
        "invalid_label_fraction": invalid_label_fraction,
        "max_jsd": max_jsd,
        "mean_category_coverage": mean_coverage,
        "structural_violation_fraction": structural_violation_fraction,
    }

    return scores, raw

def prepare_cifar10_no_leakage(
    split_seed: int,
    partition_seed: int,
    n_clients: int = 10,
    alpha: float = 1.0,
):
    (x_train_official, y_train_official), (x_test, y_test) = \
        keras.datasets.cifar10.load_data()

    y_train_official = y_train_official.reshape(-1).astype(np.int32)
    y_test = y_test.reshape(-1).astype(np.int32)

    # Official TEST is untouched. Create TRAIN/VALIDATION only from official TRAIN.
    idx = np.arange(len(x_train_official))
    train_idx, val_idx = train_test_split(
        idx,
        test_size=0.10,
        stratify=y_train_official,
        random_state=int(split_seed),
    )

    x_train = x_train_official[train_idx]
    y_train = y_train_official[train_idx]
    x_val = x_train_official[val_idx]
    y_val = y_train_official[val_idx]

    parts = dirichlet_partition_indices(
        y_train, n_clients=n_clients, alpha=alpha, seed=partition_seed
    )
    client_ids = list(parts.keys())

    client_raw = {
        cid: (x_train[idxs], y_train[idxs])
        for cid, idxs in parts.items()
    }

    mean, std = cifar_train_only_mean_std(client_raw)
    reference = cifar_reference_histograms(client_raw)

    dq_scores = {}
    dq_rows = []
    client_arrays = {}
    client_train_ids = {}

    for cid, idxs in parts.items():
        Xraw, yraw = client_raw[cid]
        scores, raw_metrics = cifar_dq_scores(Xraw, yraw, reference)
        dq_scores[cid] = scores
        dq_rows.append({"client": cid, **scores, **raw_metrics})

        X = (Xraw.astype(np.float32) / 255.0 - mean) / std
        client_arrays[cid] = (X.astype(np.float32), yraw.astype(np.int32))
        # IDs are original official-TRAIN row indices.
        client_train_ids[cid] = train_idx[idxs].astype(str)

    X_val = (x_val.astype(np.float32) / 255.0 - mean) / std
    X_test = (x_test.astype(np.float32) / 255.0 - mean) / std

    y_train_all = np.concatenate(
        [client_arrays[c][1] for c in client_ids]
    )
    cw = class_weight_dict(y_train_all)

    # TEST IDs are namespaced so overlap checks are explicit and impossible to alias.
    global_train_ids = np.array([f"official_train_{i}" for i in train_idx], dtype=object)
    client_ids_namespaced = {
        cid: np.array([f"official_train_{i}" for i in train_idx[idxs]], dtype=object)
        for cid, idxs in parts.items()
    }
    global_test_ids = np.array(
        [f"official_test_{i}" for i in range(len(x_test))], dtype=object
    )

    return {
        "client_arrays": client_arrays,
        "client_ids": client_ids,
        "X_val": X_val.astype(np.float32),
        "y_val": y_val.astype(np.int32),
        "X_test": X_test.astype(np.float32),
        "y_test": y_test.astype(np.int32),
        "dq_scores": dq_scores,
        "dq_audit": pd.DataFrame(dq_rows),
        "class_weights": cw,
        "mean": mean,
        "std": std,
        "meta": {
            "official_train_rows": int(len(x_train_official)),
            "federated_train_rows": int(len(x_train)),
            "validation_rows": int(len(x_val)),
            "official_test_rows": int(len(x_test)),
            "n_clients": int(n_clients),
        },
        "client_train_ids": client_ids_namespaced,
        "global_train_ids": global_train_ids,
        "global_test_ids": global_test_ids,
        "client_raw": client_raw,
    }


def build_cifar_model(lr: float = 1e-3) -> keras.Model:
    inp = keras.Input(shape=(32, 32, 3), dtype=tf.float32)

    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D(2)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(128, activation="relu")(x)
    out = layers.Dense(10, activation="softmax", dtype=tf.float32)(x)

    model = keras.Model(inp, out)
    model.optimizer = keras.optimizers.Adam(learning_rate=float(lr))
    return model



# ======================================================================================
# FULL CROSS-DOMAIN REPORTING, CHECKPOINTING, LEDGER, AND STATISTICS
# ======================================================================================
from datetime import datetime, timezone
import re

MANUSCRIPT_SCENARIOS = [
    "Naïve Centralized",
    "Great Expectations Centralized",
    "TADP-AA Centralized",
    "TADP-VR Centralized",
    "TADP-SDA Centralized",
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "Great Expectations Federated",
    "TADP-AA Federated",
    "TADP-VR Federated",
    "TADP-SDA Federated",
]
assert len(MANUSCRIPT_SCENARIOS) == 12


def print_banner(title: str, width: int = 108):
    print("\n" + "=" * width)
    print(title)
    print("=" * width)


def client_partition_table(clients_raw):
    rows = []
    for cid, df in clients_raw.items():
        counts = df["_target"].value_counts().to_dict()
        rows.append({
            "client": cid,
            "records": int(len(df)),
            "class_0_NO": int(counts.get(0, 0)),
            "class_1_GT30": int(counts.get(1, 0)),
            "class_2_LT30": int(counts.get(2, 0)),
        })
    return pd.DataFrame(rows)


def evidence_assignment_summary(
    evidence_df: pd.DataFrame
) -> pd.DataFrame:
    """
    Summarize the frozen controlled documentary-evidence assignment.

    v16.7 reports the pre-specified governance archetype for each evidence
    bundle. These archetypes are branch-coverage scenarios, not observed
    real-world prevalence classes.
    """
    required = {
        "client",
        "bundle_id",
        "evidence_profile",
        "scenario_role",
        "profile_variant",
        "factor",
        "rubric_score_0_5",
        "meets_factor_adequacy",
        "evidence_seed",
    }

    missing = sorted(
        required - set(evidence_df.columns)
    )

    if missing:
        raise RuntimeError(
            "Controlled documentary-evidence table is missing required "
            f"column(s): {missing}. Available columns: "
            f"{sorted(evidence_df.columns.tolist())}"
        )

    work = evidence_df.copy()
    work["meets_factor_adequacy"] = (
        work["meets_factor_adequacy"]
        .astype(bool)
    )

    summary = (
        work
        .groupby(
            ["client", "bundle_id"],
            as_index=False,
        )
        .agg(
            evidence_profile=(
                "evidence_profile",
                "first",
            ),
            scenario_role=(
                "scenario_role",
                "first",
            ),
            profile_variant=(
                "profile_variant",
                "first",
            ),
            documentary_factor_count=(
                "factor",
                "count",
            ),
            documentary_adequate_factor_count=(
                "meets_factor_adequacy",
                "sum",
            ),
            documentary_mean_score=(
                "rubric_score_0_5",
                "mean",
            ),
            documentary_min_score=(
                "rubric_score_0_5",
                "min",
            ),
            documentary_max_score=(
                "rubric_score_0_5",
                "max",
            ),
            evidence_seed=(
                "evidence_seed",
                "first",
            ),
        )
        .sort_values("client")
        .reset_index(drop=True)
    )

    summary["documentary_adequacy_fraction"] = (
        summary["documentary_adequate_factor_count"]
        / summary["documentary_factor_count"].clip(lower=1)
    )

    expected_documentary_factors = sum(
        len(FACTOR_NAMES[d])
        for d in DOCUMENTARY_DIMS
    )

    count_ok = summary[
        "documentary_factor_count"
    ].eq(
        expected_documentary_factors
    )

    if not count_ok.all():
        bad = summary.loc[
            ~count_ok,
            [
                "client",
                "bundle_id",
                "documentary_factor_count",
            ],
        ]

        raise RuntimeError(
            "Unexpected controlled-evidence factor count. "
            f"Expected {expected_documentary_factors} documentary factors "
            "per client. Offending rows:\n"
            + bad.to_string(index=False)
        )

    return summary

def print_governance_details(
    gov: pd.DataFrame,
    ge: pd.DataFrame,
    dq: pd.DataFrame,
    vr_clients: List[str],
    sda_clients: List[str],
    ge_clients: List[str],
    domain: str,
):
    domain = str(domain).lower()

    print_banner(
        "DOMAIN ADEQUACY POLICY — FULL WAC + CRITICAL WAC"
    )

    policy_rows = []
    for dim in FACTOR_NAMES:
        policy_rows.append({
            "dimension":
                dim,
            "dimension_name":
                DIMENSION_NAMES[dim],
            "n_factors":
                len(FACTOR_NAMES[dim]),
            "minimum_adequacy_ranks":
                ",".join(
                    str(
                        DOMAIN_FACTOR_MINIMA[
                            domain
                        ][dim][f]
                    )
                    for f in FACTOR_NAMES[
                        dim
                    ]
                ),
            "dimension_policy_wac":
                DOMAIN_DIMENSION_WAC[
                    domain
                ][dim],
        })

    print(
        pd.DataFrame(
            policy_rows
        ).to_string(
            index=False
        )
    )

    global_critical_wac = derive_global_critical_wac(
        domain
    )

    print(
        f"\nGLOBAL {domain.upper()} WAC "
        f"(descriptive full-policy summary) = "
        f"{DOMAIN_GLOBAL_WAC[domain]:.6f}"
    )
    print(
        f"GLOBAL {domain.upper()} CRITICAL WAC "
        f"(automated Review threshold) = "
        f"{global_critical_wac:.6f}"
    )

    print_banner(
        "CRITICAL FACTORS — INDIVIDUAL ADEQUACY REQUIREMENTS"
    )

    critical_rows = []

    for dim, factor_list in (
        CRITICAL_FACTORS_BY_DOMAIN[
            domain
        ].items()
    ):
        for factor in factor_list:
            minimum = float(
                DOMAIN_FACTOR_MINIMA[
                    domain
                ][dim][factor]
            )
            critical_rows.append({
                "dimension":
                    dim,
                "factor":
                    factor,
                "individual_adequacy_min_0_5":
                    minimum,
                "normalized_reference":
                    minimum / MAX_FACTOR_SCORE,
                "direct_auto_accept_rule":
                    f"score >= {minimum:.1f}",
            })

    print(
        pd.DataFrame(
            critical_rows
        ).to_string(
            index=False
        )
    )

    print(
        f"\nMinimum dimension floor: EVERY averaged dimension "
        f"must be >= {DIMENSION_MIN_FLOOR:.1f}/5."
    )

    print(
        "Human reviewer role: verify uploaded questionnaire evidence only. "
        "The server makes the admission decision automatically."
    )

    print_banner(
        "TRAIN-ONLY DATA-QUALITY FACTORS — 8 FACTORS"
    )

    dq_cols = [
        "client",
        "completeness",
        "duplication_rate",
        "value_validity_error_rate",
        "type_consistency",
        "label_integrity",
        "feature_distribution_consistency",
        "feature_category_coverage",
        "structural_constraint_integrity",
    ]

    print(
        dq[dq_cols].to_string(
            index=False
        )
    )

    print_banner(
        "FROZEN TADP GOVERNANCE — HPS + CRITICAL WAC"
    )

    display_cols = [
        "client",
        "hps",
        "critical_wac_i",
        "global_critical_wac",
        "critical_wac_margin",
        "all_critical_meet_adequacy",
        "critical_below_adequacy",
        "dimension_floor_failures",
        "dim1_score_0_5",
        "dim2_score_0_5",
        "dim3_score_0_5",
        "dim4_score_0_5",
        "dim5_score_0_5",
        "dim6_score_0_5",
        "decision_path",
        "initial_action",
        "final_action",
        "status",
        "reason",
    ]

    print(
        gov[
            display_cols
        ].to_string(
            index=False
        )
    )

    print(
        "\nTADP v16.7 final decision order:"
    )
    print(
        f"  1) ANY averaged dimension < "
        f"{DIMENSION_MIN_FLOOR:.1f}/5 -> AUTO-REJECT"
    )
    print(
        f"  2) HPS < {GOOD_CUT:.1f} -> AUTO-REJECT"
    )
    print(
        f"  3) HPS >= {HIGH_CUT:.1f}:"
    )
    print(
        "       all critical factors meet their own adequacy minima "
        "-> DIRECT AUTO-ACCEPT"
    )
    print(
        "       otherwise -> AUTOMATED REVIEW fallback"
    )
    print(
        f"  4) {GOOD_CUT:.1f} <= HPS < "
        f"{HIGH_CUT:.1f} -> AUTOMATED REVIEW"
    )
    print(
        "  5) AUTOMATED REVIEW:"
    )
    print(
        f"       Critical WAC_i >= Global Critical WAC "
        f"({global_critical_wac:.6f}) -> ACCEPT AFTER REVIEW"
    )
    print(
        "       otherwise -> AUTO-REJECT"
    )

    print(
        f"\nFrozen TADP-VR cohort: "
        f"{len(vr_clients)}/10 -> "
        f"{vr_clients}"
    )
    print(
        f"Frozen TADP-SDA cohort: "
        f"{len(sda_clients)}/10 -> "
        f"{sda_clients}"
    )

    print_banner(
        "FROZEN GX CORE TRAIN-ONLY MATCHED-K COMPARATOR"
    )

    gx_base_cols = [
        "client",
        "gx_native_suite_success",
        "ge_expectations_passed",
        "ge_expectations_total",
        "ge_pass_rate",
        "ge_critical_failures",
        "ge_warning_failures",
        "ge_mean_unexpected_percent",
        "ge_native_validation_class",
        "ge_rank",
        "ge_final_action",
    ]

    gx_category_cols = [
        c
        for c in ge.columns
        if c.startswith("ge_")
        and (
            c.endswith("_passed")
            or c.endswith("_total")
        )
        and c not in {
            "ge_expectations_passed",
            "ge_expectations_total",
        }
    ]

    gx_display_cols = (
        gx_base_cols
        + sorted(gx_category_cols)
    )

    print(
        ge[
            gx_display_cols
        ].to_string(
            index=False
        )
    )

    print(
        f"\nGX Core matched-K selected "
        f"{len(ge_clients)}/{len(ge)} -> "
        f"{ge_clients}"
    )

    print(
        "GX native validation results are reported separately. "
        "The experiment's ACCEPT/REJECT column is a matched-K lexicographic "
        "selection used only to equalize downstream cohort size with TADP-VR. "
        "Critical failures are retained as the strongest ranking criterion, "
        "not used as a hard eligibility filter."
    )



def build_hash_chained_governance_ledger(
    gov,
    ge,
    output_path,
):
    rows = []
    prev_hash = "GENESIS"
    seq = 0

    for _, r in (
        gov.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "TADP",
            "client":
                str(r["client"]),
            "hps":
                float(r["hps"]),
            "global_domain_wac":
                float(r["global_domain_wac"]),
            "global_critical_wac":
                float(r["global_critical_wac"]),
            "critical_wac_i": (
                float(r["critical_wac_i"])
                if np.isfinite(r["critical_wac_i"])
                else None
            ),
            "critical_wac_margin": (
                float(r["critical_wac_margin"])
                if np.isfinite(r["critical_wac_margin"])
                else None
            ),
            "all_critical_meet_adequacy":
                bool(r["all_critical_meet_adequacy"]),
            "critical_below_adequacy":
                str(r["critical_below_adequacy"]),
            "dimension_floor_failures":
                str(r["dimension_floor_failures"]),
            "decision_path":
                str(r["decision_path"]),
            "initial_action":
                str(r["initial_action"]),
            "final_action":
                str(r["final_action"]),
            "status":
                str(r["status"]),
            "reason":
                str(r["reason"]),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    for _, r in (
        ge.sort_values(
            "client"
        ).iterrows()
    ):
        seq += 1

        payload = {
            "sequence":
                seq,
            "governance_system":
                "Great Expectations GX Core matched-K comparator",
            "client":
                str(r["client"]),
            "hps":
                None,
            "global_domain_wac":
                None,
            "global_critical_wac":
                None,
            "critical_wac_i":
                None,
            "critical_wac_margin":
                None,
            "all_critical_meet_adequacy":
                None,
            "critical_below_adequacy":
                None,
            "dimension_floor_failures":
                None,
            "decision_path":
                "RULE_BASED_VALIDATION",
            "initial_action":
                "RULE_BASED_VALIDATION",
            "final_action":
                str(r["ge_final_action"]),
            "status":
                "GE_CONTROLLED_TRAIN_ONLY",
            "reason": (
                f"{int(r['ge_expectations_passed'])}/"
                f"{int(r['ge_expectations_total'])} "
                "GX Core Expectations passed; "
                f"critical_failures={int(r['ge_critical_failures'])}; "
                f"warning_failures={int(r['ge_warning_failures'])}; "
                f"rank={int(r['ge_rank'])}"
            ),
            "previous_hash":
                prev_hash,
        }

        canonical = json.dumps(
            payload,
            sort_keys=True,
            separators=(",", ":"),
        )

        entry_hash = hashlib.sha256(
            canonical.encode("utf-8")
        ).hexdigest()

        payload["entry_hash"] = entry_hash

        rows.append(payload)
        prev_hash = entry_hash

    out = pd.DataFrame(rows)

    out.to_csv(
        output_path,
        index=False,
    )

    return out

def choose_experiment_root(experiment_name, use_drive=True):
    if use_drive:
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
            root = Path("/content/drive/MyDrive/TADP_CHECKPOINTS") / experiment_name
            root.mkdir(parents=True, exist_ok=True)
            print(f"Persistent checkpoint root: {root}")
            return root
        except Exception as exc:
            print(f"Google Drive checkpoint mount unavailable: {exc}")
    root = Path("/content") / experiment_name
    root.mkdir(parents=True, exist_ok=True)
    print(f"Local checkpoint root: {root}")
    return root


def atomic_write_json(obj, path):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")
    tmp.replace(path)


def load_checkpoint_state(path):
    path = Path(path)
    if not path.exists():
        return {"completed": [], "last_completed": None, "updated_utc": None}
    return json.loads(path.read_text(encoding="utf-8"))


def mark_checkpoint_complete(state_path, key, extra=None):
    state = load_checkpoint_state(state_path)
    completed = list(state.get("completed", []))
    if key not in completed:
        completed.append(key)
    state["completed"] = completed
    state["last_completed"] = key
    state["updated_utc"] = datetime.now(timezone.utc).isoformat()
    if extra:
        state.update(extra)
    atomic_write_json(state, state_path)


def upsert_csv(row, path, key_cols):
    path = Path(path)
    new = pd.DataFrame([row])
    if path.exists():
        old = pd.read_csv(path)
        if not old.empty:
            mask = pd.Series(True, index=old.index)
            for c in key_cols:
                mask &= old[c].astype(str).eq(str(row[c]))
            old = old.loc[~mask].copy()
            new = pd.concat([old, new], ignore_index=True)
    new.to_csv(path, index=False)


def write_scenario_checkpoint(root, run_idx, scenario, result):
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", scenario).strip("_")
    cdir = ensure_dir(root / "scenario_checkpoints" / f"run_{run_idx:02d}")
    payload = {
        "run": int(run_idx),
        "scenario": scenario,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
        "metrics": result["metrics"],
        "runtime_s": float(result["runtime_s"]),
        "optimizer_steps": int(result["total_optimizer_steps"]),
        "communication_mb": float(result.get("communication_mb", 0.0)),
        "ram_peak_mb": float(result.get("ram_peak_mb", 0.0)),
    }
    atomic_write_json(payload, cdir / f"{safe}.json")
    if "round_audit" in result:
        pd.DataFrame(result["round_audit"]).to_csv(
            cdir / f"{safe}_rounds.csv", index=False
        )


def ci95_mean(values):
    x = np.asarray(values, dtype=float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan
    if len(x) == 1:
        return float(x[0]), float(x[0])
    mean = float(np.mean(x))
    sd = float(np.std(x, ddof=1))
    try:
        from scipy.stats import t
        crit = float(t.ppf(0.975, df=len(x)-1))
    except Exception:
        crit = 1.96
    half = crit * sd / math.sqrt(len(x))
    return mean-half, mean+half


def summarize_runs_with_ci(perf):
    metrics = [
        "accuracy", "precision_macro", "recall_macro", "f1_macro",
        "roc_auc_ovr_macro", "runtime_s", "energy_wh", "energy_kwh",
        "co2_kg", "communication_mb", "ram_peak_mb", "ram_delta_mb",
        "optimizer_steps", "participants", "energy_cost_usd",
        "communication_cost_usd", "total_estimated_cost_usd",
    ]
    rows = []
    for scenario, d in perf.groupby("scenario", sort=False):
        row = {"scenario": scenario, "n_runs": int(len(d))}
        for metric in metrics:
            if metric not in d.columns:
                continue
            vals = pd.to_numeric(d[metric], errors="coerce")
            vals = vals[np.isfinite(vals)]
            row[f"{metric}_mean"] = float(vals.mean()) if len(vals) else np.nan
            row[f"{metric}_sd"] = float(vals.std(ddof=1)) if len(vals)>1 else 0.0
            lo, hi = ci95_mean(vals)
            row[f"{metric}_ci95_low"] = lo
            row[f"{metric}_ci95_high"] = hi
        rows.append(row)
    return pd.DataFrame(rows)


def paired_vr_randomk_statistics(perf):
    vr = perf[perf["scenario"].eq("TADP-VR Federated")].copy()
    rk = perf[perf["scenario"].eq("Random-K")].copy()
    merged = vr.merge(rk, on=["run", "seed"], suffixes=("_vr", "_randomk"), validate="one_to_one")
    rows = []
    for metric in ["accuracy", "f1_macro", "roc_auc_ovr_macro"]:
        a = merged[f"{metric}_vr"].to_numpy(dtype=float)
        b = merged[f"{metric}_randomk"].to_numpy(dtype=float)
        diff = a-b
        mean = float(np.mean(diff))
        sd = float(np.std(diff, ddof=1)) if len(diff)>1 else 0.0
        lo, hi = ci95_mean(diff)
        t_stat=t_p=wil_stat=wil_p=np.nan
        try:
            from scipy.stats import ttest_rel, wilcoxon
            tr = ttest_rel(a,b,nan_policy="omit")
            t_stat, t_p = float(tr.statistic), float(tr.pvalue)
            if np.any(np.abs(diff)>0):
                wr = wilcoxon(a,b)
                wil_stat, wil_p = float(wr.statistic), float(wr.pvalue)
        except Exception:
            pass
        rows.append({
            "metric": metric,
            "n_pairs": len(diff),
            "mean_difference_vr_minus_randomk": mean,
            "sd_difference": sd,
            "ci95_low": lo,
            "ci95_high": hi,
            "cohens_dz": float(mean/sd) if sd>0 else np.nan,
            "paired_t_stat": t_stat,
            "paired_t_p": t_p,
            "wilcoxon_stat": wil_stat,
            "wilcoxon_p": wil_p,
            "vr_wins": int(np.sum(diff>0)),
            "ties": int(np.sum(np.isclose(diff,0))),
            "vr_losses": int(np.sum(diff<0)),
        })
    return pd.DataFrame(rows)


def scenario_method_table():
    return pd.DataFrame([
        ["Naïve Centralized","centralized","baseline","all clients"],
        ["Great Expectations Centralized","centralized","GE","frozen GE cohort"],
        ["TADP-AA Centralized","centralized","TADP-AA","all clients"],
        ["TADP-VR Centralized","centralized","TADP-VR","frozen TADP-VR cohort"],
        ["TADP-SDA Centralized","centralized","TADP-SDA","frozen best eligible client"],
        ["Vanilla FedAvg","federated","FedAvg","all clients"],
        ["FedProx","federated","FedProx","all clients"],
        ["Random-K","federated","matched control","frozen Random-K cohort"],
        ["Great Expectations Federated","federated","GE","frozen GE cohort"],
        ["TADP-AA Federated","federated","TADP-AA","all clients"],
        ["TADP-VR Federated","federated","TADP-VR","frozen TADP-VR cohort"],
        ["TADP-SDA Federated","federated","TADP-SDA","frozen best eligible client"],
    ], columns=["scenario","paradigm","method","participation"])


def governance_only_monte_carlo(
    client_ids,
    dq_scores,
    base_seed,
    n_realizations=1000,
    domain="healthcare",
):
    domain = str(domain).lower()
    rows=[]
    for j in range(int(n_realizations)):
        seed=int(base_seed+j)
        evidence,_=generate_controlled_documentary_evidence(
            client_ids,
            seed,
            DOMAIN_FACTOR_MINIMA[domain],
            domain=domain,
        )
        gov=build_tadp_governance(
            client_ids,
            evidence,
            dq_scores,
            run=0,
            evidence_seed=seed,
            domain=domain,
        )
        accepted=accepted_tadp_vr(gov)
        rows.append({
            "realization":j+1,
            "evidence_seed":seed,
            "accepted_count":len(accepted),
            "accepted_clients":";".join(accepted),
            "mean_hps":float(gov["hps"].mean()),
            "mean_critical_wac_i":float(gov["critical_wac_i"].mean()),
            "dimension_floor_rejects":int(gov["status"].eq("AUTO_REJECTED_DIMENSION_FLOOR").sum()),
            "low_hps_rejects":int(gov["status"].eq("AUTO_REJECTED_LOW_HPS").sum()),
            "review_accepts":int(
                gov["status"].eq("ACCEPTED_AFTER_AUTOMATED_REVIEW").sum()
            ),
            "direct_auto_accepts":int(
                gov["status"].eq("DIRECT_AUTO_ACCEPTED").sum()
            ),
        })
    return pd.DataFrame(rows)



def cifar_partition_table(
    client_arrays: Dict[str, Tuple[np.ndarray, np.ndarray]]
) -> pd.DataFrame:
    rows = []

    for cid, (_, y) in client_arrays.items():
        counts = np.bincount(
            np.asarray(y).reshape(-1),
            minlength=10,
        )

        row = {
            "client": cid,
            "records": int(len(y)),
        }
        for k in range(10):
            row[f"class_{k}"] = int(counts[k])

        rows.append(row)

    return pd.DataFrame(rows)




# ======================================================================================
# EXPERIMENT C v16.7 — CIFAR-10 CROSS-DOMAIN GOVERNANCE
# ======================================================================================

EXPERIMENT_VERSION = "TADP-CIFAR10-v16.7-FULL-NOLEAK-GXCORE-MATCHEDKFIX-CRITICALWAC-28F"

TRAINING_RUN_SEEDS = [42, 142, 242]
NUM_CLIENTS = 10
NUM_ROUNDS_FL = 10
LOCAL_EPOCHS = 1
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
FEDPROX_MU = 0.01
DIRICHLET_ALPHA = 1.0

GLOBAL_SPLIT_SEED = 8101
CLIENT_PARTITION_SEED = 8201

# One-time controls — NOT experimental repetitions.
FROZEN_EVIDENCE_ASSIGNMENT_SEED = 3042
FROZEN_RANDOMK_SELECTION_SEED = 4042

USE_GOOGLE_DRIVE_CHECKPOINTS = True

CIFAR_SCENARIOS = [
    "Vanilla FedAvg",
    "FedProx",
    "Random-K",
    "Great Expectations Federated",
    "TADP-AA Federated",
    "TADP-VR Federated",
    "TADP-SDA Federated",
]
assert len(CIFAR_SCENARIOS) == 7

EXPERIMENT_ROOT = choose_experiment_root(
    "TADP_EXPERIMENT_C_CIFAR10_v16_7_GXCORE_MATCHEDKFIX",
    use_drive=USE_GOOGLE_DRIVE_CHECKPOINTS,
)
CHECKPOINT_STATE = EXPERIMENT_ROOT / "checkpoint_state.json"
PERF_CHECKPOINT = EXPERIMENT_ROOT / "performance_metrics_checkpoint.csv"


def main():
    print_banner(EXPERIMENT_VERSION)
    print(f"Experimental training seeds: {TRAINING_RUN_SEEDS}")
    print(
        f"Each training seed executes ALL {len(CIFAR_SCENARIOS)} "
        "CIFAR federated scenarios."
    )
    print(
        f"Total trained configurations: "
        f"{len(TRAINING_RUN_SEEDS) * len(CIFAR_SCENARIOS)}"
    )
    print(f"FL rounds: {NUM_ROUNDS_FL}")
    print(f"Local epochs: {LOCAL_EPOCHS}")
    print(f"Batch size: {BATCH_SIZE}")
    print(f"Dirichlet alpha: {DIRICHLET_ALPHA}")
    print(
        f"One-time evidence-assignment RNG seed: "
        f"{FROZEN_EVIDENCE_ASSIGNMENT_SEED} "
        "[not an experimental run seed]"
    )
    print(
        f"One-time Random-K selection RNG seed: "
        f"{FROZEN_RANDOMK_SELECTION_SEED} "
        "[not an experimental run seed]"
    )
    print(
        f"Global CIFAR-10 reference WAC: "
        f"{WAC_CIFAR10:.6f} "
        "(derived from 28 factor-specific adequacy ranks)"
    )

    # ------------------------------------------------------------------
    # 1) Official TEST remains untouched.
    # ------------------------------------------------------------------
    data = prepare_cifar10_no_leakage(
        split_seed=GLOBAL_SPLIT_SEED,
        partition_seed=CLIENT_PARTITION_SEED,
        n_clients=NUM_CLIENTS,
        alpha=DIRICHLET_ALPHA,
    )

    leakage = write_leakage_audit(
        EXPERIMENT_ROOT,
        data["global_train_ids"],
        data["global_test_ids"],
        data["client_train_ids"],
        extra={
            "official_test_untouched_before_final_evaluation": True,
            "validation_excluded_from_client_partition": True,
            "normalization_train_only": True,
            "dq_ge_tadp_train_only": True,
            "class_weights_train_only": True,
        },
    )

    print_banner("NO-LEAKAGE AUDIT — MUST PASS")
    print(pd.DataFrame([leakage]).to_string(index=False))

    pd.DataFrame([{
        **data["meta"],
        "train_rgb_mean": data["mean"].tolist(),
        "train_rgb_std": data["std"].tolist(),
    }]).to_json(
        EXPERIMENT_ROOT / "dataset_metadata.json",
        orient="records",
        indent=2,
    )

    partition_df = cifar_partition_table(
        data["client_arrays"]
    )
    partition_df.to_csv(
        EXPERIMENT_ROOT / "client_partition_audit.csv",
        index=False,
    )

    print_banner("FROZEN CIFAR-10 TRAIN CLIENT PARTITIONS")
    print(partition_df.to_string(index=False))

    data["dq_audit"].to_csv(
        EXPERIMENT_ROOT / "dq_train_only_audit.csv",
        index=False,
    )

    client_ids = list(data["client_ids"])
    client_arrays = data["client_arrays"]
    X_test = data["X_test"]
    y_test = data["y_test"]
    class_weights = data["class_weights"]

    build_model_fn = lambda: build_cifar_model(
        LEARNING_RATE
    )

    # ------------------------------------------------------------------
    # 2) Governance generated ONCE and frozen.
    # ------------------------------------------------------------------
    documentary_evidence, evidence_df = (
        generate_controlled_documentary_evidence(
            client_ids,
            FROZEN_EVIDENCE_ASSIGNMENT_SEED,
            FACTOR_ADEQUACY_MIN_CIFAR10,
            domain="cifar10",
        )
    )

    evidence_df.to_csv(
        EXPERIMENT_ROOT / "controlled_documentary_evidence_matrix.csv",
        index=False,
    )

    evidence_summary = evidence_assignment_summary(
        evidence_df
    )
    evidence_summary.to_csv(
        EXPERIMENT_ROOT / "evidence_assignment_summary.csv",
        index=False,
    )

    print_banner("FROZEN CONTROLLED DOCUMENTARY EVIDENCE — CIFAR-10 — SCHEMA VERIFIED")
    print(evidence_summary.to_string(index=False))

    gov = build_tadp_governance(
        client_ids,
        documentary_evidence,
        data["dq_scores"],
        run=0,
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain="cifar10",
    )
    gov.to_csv(
        EXPERIMENT_ROOT /
        "tadp_hps_client_adequacy_governance_frozen.csv",
        index=False,
    )

    full_evidence = build_full_factor_evidence_table(
        client_ids=client_ids,
        documentary_evidence=documentary_evidence,
        dq_scores=data["dq_scores"],
        dq_audit=data["dq_audit"],
        evidence_seed=FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        domain="cifar10",
    )
    full_evidence.to_csv(
        EXPERIMENT_ROOT /
        "all_28_factor_evidence_by_client.csv",
        index=False,
    )

    pd.DataFrame([
        {
            "domain": "cifar10",
            "dimension": dim,
            "dimension_name": DIMENSION_NAMES[dim],
            "factor_count": len(FACTOR_NAMES[dim]),
            "adequacy_ranks": ";".join(
                str(FACTOR_ADEQUACY_MIN_CIFAR10[dim][f])
                for f in FACTOR_NAMES[dim]
            ),
            "dimension_policy_wac":
                CIFAR10_DIMENSION_WAC[dim],
            "global_domain_wac": WAC_CIFAR10,
            "global_critical_wac": derive_global_critical_wac("cifar10"),
        }
        for dim in FACTOR_NAMES
    ]).to_csv(
        EXPERIMENT_ROOT /
        "cifar10_wac_policy_derivation.csv",
        index=False,
    )

    all_clients = list(client_ids)
    vr_clients = accepted_tadp_vr(gov)
    sda_clients = accepted_tadp_sda(gov)

    if not vr_clients:
        raise RuntimeError(
            "CIFAR TADP-VR admitted zero clients. Fail closed."
        )

    # REAL GX Core validation on structured image metadata, matched to TADP-VR K.
    ge, ge_expectation_audit = ge_governance(
        client_data=data["client_raw"],
        client_ids=client_ids,
        accept_count=len(vr_clients),
        domain="cifar10",
        dq_scores=data["dq_scores"],
    )

    ge.to_csv(
        EXPERIMENT_ROOT /
        "gx_core_train_only_governance_frozen.csv",
        index=False,
    )
    ge_expectation_audit.to_csv(
        EXPERIMENT_ROOT /
        "gx_core_expectation_audit.csv",
        index=False,
    )

    # Backward-compatible filename retained.
    ge.to_csv(
        EXPERIMENT_ROOT /
        "ge_train_only_governance_frozen.csv",
        index=False,
    )

    ge_clients = ge.loc[
        ge["ge_final_action"].eq("ACCEPT"),
        "client",
    ].astype(str).tolist()

    rk_rng = np.random.default_rng(
        FROZEN_RANDOMK_SELECTION_SEED
    )
    randomk_clients = rk_rng.choice(
        client_ids,
        size=len(vr_clients),
        replace=False,
    ).tolist()

    if len(ge_clients) != len(vr_clients):
        raise RuntimeError(
            f"GX matched-K cohort size {len(ge_clients)} does not match "
            f"TADP-VR K={len(vr_clients)}."
        )

    if len(randomk_clients) != len(vr_clients):
        raise RuntimeError(
            "Random-K K does not match TADP-VR K."
        )

    frozen_plan = {
        "experiment_version": EXPERIMENT_VERSION,
        "training_seeds": TRAINING_RUN_SEEDS,
        "fl_rounds": NUM_ROUNDS_FL,
        "global_split_seed": GLOBAL_SPLIT_SEED,
        "client_partition_seed": CLIENT_PARTITION_SEED,
        "frozen_evidence_assignment_seed":
            FROZEN_EVIDENCE_ASSIGNMENT_SEED,
        "frozen_randomk_selection_seed":
            FROZEN_RANDOMK_SELECTION_SEED,
        "all_clients": all_clients,
        "ge_clients": ge_clients,
        "tadp_vr_clients": vr_clients,
        "tadp_sda_clients": sda_clients,
        "randomk_clients": randomk_clients,
        "global_cifar10_wac": WAC_CIFAR10,
        "cifar10_dimension_wac": CIFAR10_DIMENSION_WAC,
        "client_review_metric":
            "Client Adequacy Score (CAS_i)",
    }

    plan_path = (
        EXPERIMENT_ROOT / "frozen_cohort_plan.json"
    )
    if plan_path.exists():
        previous = json.loads(
            plan_path.read_text(encoding="utf-8")
        )
        for key in [
            "all_clients",
            "ge_clients",
            "tadp_vr_clients",
            "tadp_sda_clients",
            "randomk_clients",
            "frozen_evidence_assignment_seed",
            "frozen_randomk_selection_seed",
        ]:
            if previous.get(key) != frozen_plan.get(key):
                raise RuntimeError(
                    f"Frozen plan changed on resume for {key}: "
                    f"old={previous.get(key)}, "
                    f"new={frozen_plan.get(key)}"
                )

    atomic_write_json(
        frozen_plan,
        plan_path,
    )

    print_governance_details(
        gov,
        ge,
        data["dq_audit"],
        vr_clients,
        sda_clients,
        ge_clients,
        domain="cifar10",
    )

    print_banner("FROZEN RANDOM-K MATCHED CONTROL — CIFAR-10")
    print(
        f"TADP-VR K={len(vr_clients)} -> {vr_clients}"
    )
    print(
        f"Random-K K={len(randomk_clients)} -> {randomk_clients}"
    )
    print(
        "Both client identity sets remain unchanged across "
        "all 10 rounds and all 3 training runs."
    )

    build_hash_chained_governance_ledger(
        gov,
        ge,
        EXPERIMENT_ROOT /
        "append_only_governance_ledger.csv",
    )

    pd.DataFrame([
        ["Vanilla FedAvg", "Baseline", "all clients"],
        ["FedProx", "Baseline", "all clients"],
        ["Random-K", "Matched control", "frozen Random-K cohort"],
        ["Great Expectations Federated", "GE", "frozen GE cohort"],
        ["TADP-AA Federated", "TADP-AA", "all clients"],
        ["TADP-VR Federated", "TADP-VR", "frozen TADP-VR cohort"],
        ["TADP-SDA Federated", "TADP-SDA", "frozen best eligible client"],
    ], columns=[
        "scenario",
        "method",
        "participation",
    ]).to_csv(
        EXPERIMENT_ROOT /
        "scenario_definitions.csv",
        index=False,
    )

    # ------------------------------------------------------------------
    # 3) Frozen exact optimizer-step plans.
    # ------------------------------------------------------------------
    def natural_map(cohort):
        return {
            c: natural_steps(
                len(client_arrays[c][1]),
                BATCH_SIZE,
                LOCAL_EPOCHS,
            )
            for c in cohort
        }

    step_map_all = natural_map(all_clients)
    step_map_ge = natural_map(ge_clients)
    step_map_vr = natural_map(vr_clients)
    step_map_sda = natural_map(sda_clients)

    vr_target_per_round = int(
        sum(step_map_vr.values())
    )

    step_map_randomk = allocate_exact_step_budget(
        randomk_clients,
        client_arrays,
        vr_target_per_round,
        BATCH_SIZE,
        LOCAL_EPOCHS,
    )

    parity_rows = []

    for r in range(1, NUM_ROUNDS_FL + 1):
        parity_rows.append({
            "round": r,
            "tadp_vr_clients": ";".join(vr_clients),
            "randomk_clients": ";".join(randomk_clients),
            "tadp_vr_k": len(vr_clients),
            "randomk_k": len(randomk_clients),
            "tadp_vr_steps": vr_target_per_round,
            "randomk_steps":
                int(sum(step_map_randomk.values())),
            "same_k":
                len(vr_clients) == len(randomk_clients),
            "exact_step_parity":
                vr_target_per_round
                == int(sum(step_map_randomk.values())),
            "tadp_vr_step_map":
                json.dumps(step_map_vr, sort_keys=True),
            "randomk_step_map":
                json.dumps(step_map_randomk, sort_keys=True),
        })

    parity_df = pd.DataFrame(parity_rows)
    parity_df.to_csv(
        EXPERIMENT_ROOT /
        "frozen_vr_randomk_parity_plan.csv",
        index=False,
    )

    if not parity_df[
        ["same_k", "exact_step_parity"]
    ].all().all():
        raise RuntimeError(
            "CIFAR TADP-VR / Random-K parity plan failed."
        )

    print_banner(
        "EXACT TADP-VR vs RANDOM-K PER-ROUND PARITY PLAN"
    )
    print(parity_df.to_string(index=False))

    cohorts = {
        "Vanilla FedAvg": all_clients,
        "FedProx": all_clients,
        "Random-K": randomk_clients,
        "Great Expectations Federated": ge_clients,
        "TADP-AA Federated": all_clients,
        "TADP-VR Federated": vr_clients,
        "TADP-SDA Federated": sda_clients,
    }

    step_maps = {
        "Vanilla FedAvg": [
            dict(step_map_all)
            for _ in range(NUM_ROUNDS_FL)
        ],
        "FedProx": [
            dict(step_map_all)
            for _ in range(NUM_ROUNDS_FL)
        ],
        "Random-K": [
            dict(step_map_randomk)
            for _ in range(NUM_ROUNDS_FL)
        ],
        "Great Expectations Federated": [
            dict(step_map_ge)
            for _ in range(NUM_ROUNDS_FL)
        ],
        "TADP-AA Federated": [
            dict(step_map_all)
            for _ in range(NUM_ROUNDS_FL)
        ],
        "TADP-VR Federated": [
            dict(step_map_vr)
            for _ in range(NUM_ROUNDS_FL)
        ],
        "TADP-SDA Federated": [
            dict(step_map_sda)
            for _ in range(NUM_ROUNDS_FL)
        ],
    }

    # ------------------------------------------------------------------
    # 4) Train all scenarios x all three seeds.
    # ------------------------------------------------------------------
    completed = set(
        load_checkpoint_state(
            CHECKPOINT_STATE
        ).get("completed", [])
    )

    selection_rows = []

    for run_idx, seed in enumerate(
        TRAINING_RUN_SEEDS,
        start=1,
    ):
        print_banner(
            f"CIFAR EXPERIMENTAL RUN "
            f"{run_idx}/{len(TRAINING_RUN_SEEDS)} | "
            f"TRAINING SEED={seed} | "
            f"RUNNING ALL {len(CIFAR_SCENARIOS)} SCENARIOS | "
            "GOVERNANCE/CLIENTS FROZEN"
        )

        seed_everything(seed)
        init_model = build_model_fn()
        initial_weights = [
            np.array(w, copy=True)
            for w in init_model.get_weights()
        ]
        initial_hash = sha256_weights(
            initial_weights
        )

        del init_model
        tf.keras.backend.clear_session()
        gc.collect()

        upsert_csv(
            {
                "run": run_idx,
                "seed": seed,
                "initial_weights_sha256":
                    initial_hash,
            },
            EXPERIMENT_ROOT /
            "initialization_audit.csv",
            ["run"],
        )

        for scenario_idx, scenario in enumerate(
            CIFAR_SCENARIOS,
            start=1,
        ):
            key = (
                f"run={run_idx}|scenario={scenario}"
            )

            print_banner(
                f"CIFAR SCENARIO "
                f"{scenario_idx}/{len(CIFAR_SCENARIOS)} | "
                f"RUN {run_idx}/{len(TRAINING_RUN_SEEDS)} | "
                f"{scenario}"
            )

            if key in completed:
                print(
                    f"CHECKPOINT FOUND -> skipping completed "
                    f"scenario: {key}"
                )
                continue

            selected = list(
                cohorts[scenario]
            )

            selection_rows.append({
                "run": run_idx,
                "scenario": scenario,
                "selected_clients":
                    ";".join(selected),
                "selected_count":
                    len(selected),
                "selection_frozen_across_rounds":
                    True,
            })

            print(
                f"Selected clients (frozen): {selected}"
            )

            selected_per_round = [
                list(selected)
                for _ in range(NUM_ROUNDS_FL)
            ]
            exact_maps = step_maps[scenario]

            print(
                f"{NUM_ROUNDS_FL}-round frozen "
                "participation/step plan:"
            )

            for r in range(NUM_ROUNDS_FL):
                print(
                    f"  Round {r+1}/{NUM_ROUNDS_FL} | "
                    f"clients={selected_per_round[r]} | "
                    f"steps/client={exact_maps[r]} | "
                    f"total={sum(exact_maps[r].values())}"
                )

            seed_everything(seed)

            result = federated_train(
                build_model_fn,
                initial_weights,
                selected_per_round,
                client_arrays,
                X_test,
                y_test,
                10,
                BATCH_SIZE,
                LOCAL_EPOCHS,
                class_weights,
                run_seed=seed,
                exact_step_maps=exact_maps,
                equal_weight=False,
                fedprox_mu=(
                    FEDPROX_MU
                    if scenario == "FedProx"
                    else 0.0
                ),
            )

            row = result_row(
                run_idx,
                seed,
                scenario,
                result,
                initial_hash,
            )
            row[
                "selected_client_ids"
            ] = ";".join(selected)
            row[
                "selected_client_count"
            ] = len(selected)
            row[
                "fl_rounds"
            ] = NUM_ROUNDS_FL
            row[
                "global_cifar10_wac"
            ] = WAC_CIFAR10
            row[
                "frozen_evidence_assignment_seed"
            ] = FROZEN_EVIDENCE_ASSIGNMENT_SEED

            upsert_csv(
                row,
                PERF_CHECKPOINT,
                ["run", "scenario"],
            )

            write_scenario_checkpoint(
                EXPERIMENT_ROOT,
                run_idx,
                scenario,
                result,
            )

            mark_checkpoint_complete(
                CHECKPOINT_STATE,
                key,
                extra={
                    "experiment_version":
                        EXPERIMENT_VERSION,
                    "run": run_idx,
                    "scenario": scenario,
                    "completed_units":
                        len(completed) + 1,
                    "total_units":
                        len(TRAINING_RUN_SEEDS)
                        * len(CIFAR_SCENARIOS),
                },
            )
            completed.add(key)

            print("\nFINAL HELD-OUT OFFICIAL TEST METRICS:")
            print(
                f"  Accuracy={row['accuracy']:.6f} | "
                f"Precision={row['precision_macro']:.6f} | "
                f"Recall={row['recall_macro']:.6f} | "
                f"F1={row['f1_macro']:.6f} | "
                f"AUC={row['roc_auc_ovr_macro']:.6f}"
            )
            print(
                f"  Runtime={row['runtime_s']:.2f}s | "
                f"Energy={row['energy_wh']:.6f}Wh "
                f"({row['energy_kwh']:.9f}kWh) | "
                f"CO2={row['co2_kg']:.9f}kg"
            )
            print(
                f"  Communication="
                f"{row['communication_mb']:.3f}MB | "
                f"RAM start={row['ram_start_mb']:.1f}MB | "
                f"RAM peak={row['ram_peak_mb']:.1f}MB | "
                f"RAM delta={row['ram_delta_mb']:.1f}MB"
            )
            print(
                f"  Energy cost="
                f"${row['energy_cost_usd']:.8f} | "
                f"Communication cost="
                f"${row['communication_cost_usd']:.8f} | "
                f"Total estimated cost="
                f"${row['total_estimated_cost_usd']:.8f}"
            )
            print(
                f"  Optimizer steps="
                f"{row['optimizer_steps']} | "
                f"selected clients="
                f"{row['selected_client_count']}"
            )
            print(
                f"✅ SCENARIO DONE: {scenario} | "
                f"run={run_idx}/"
                f"{len(TRAINING_RUN_SEEDS)}"
            )
            print(
                f"CHECKPOINT SAVED under: "
                f"{EXPERIMENT_ROOT / 'scenario_checkpoints'}"
            )

            del result["model"]
            tf.keras.backend.clear_session()
            gc.collect()

    # ------------------------------------------------------------------
    # 5) Final fail-closed audits / publication outputs.
    # ------------------------------------------------------------------
    perf = pd.read_csv(
        PERF_CHECKPOINT
    )

    expected = (
        len(TRAINING_RUN_SEEDS)
        * len(CIFAR_SCENARIOS)
    )

    if len(
        perf[
            ["run", "scenario"]
        ].drop_duplicates()
    ) != expected:
        raise RuntimeError(
            f"Incomplete CIFAR run: expected {expected} "
            "unique run-scenario results."
        )

    init_rows = []

    for run_idx, d in perf.groupby("run"):
        n_hash = d[
            "initial_weights_sha256"
        ].nunique()

        init_rows.append({
            "run": int(run_idx),
            "scenario_count": int(len(d)),
            "unique_initial_weight_hashes":
                int(n_hash),
            "initialization_parity_pass":
                bool(n_hash == 1),
        })

    init_audit = pd.DataFrame(init_rows)
    init_audit.to_csv(
        EXPERIMENT_ROOT /
        "initialization_parity_audit.csv",
        index=False,
    )

    if not init_audit[
        "initialization_parity_pass"
    ].all():
        raise RuntimeError(
            "CIFAR initialization parity failed."
        )

    vr_sel = perf.loc[
        perf["scenario"].eq(
            "TADP-VR Federated"
        ),
        "selected_client_ids",
    ].astype(str).unique()

    rk_sel = perf.loc[
        perf["scenario"].eq(
            "Random-K"
        ),
        "selected_client_ids",
    ].astype(str).unique()

    frozen_audit = pd.DataFrame([{
        "tadp_vr_unique_selections_across_runs":
            len(vr_sel),
        "randomk_unique_selections_across_runs":
            len(rk_sel),
        "tadp_vr_frozen_across_runs":
            len(vr_sel) == 1,
        "randomk_frozen_across_runs":
            len(rk_sel) == 1,
        "tadp_vr_clients":
            vr_sel[0] if len(vr_sel) else "",
        "randomk_clients":
            rk_sel[0] if len(rk_sel) else "",
    }])

    frozen_audit.to_csv(
        EXPERIMENT_ROOT /
        "frozen_cohort_audit.csv",
        index=False,
    )

    matched = []

    for run_idx in sorted(
        perf["run"].unique()
    ):
        vr = perf[
            (perf["run"].eq(run_idx))
            & perf["scenario"].eq(
                "TADP-VR Federated"
            )
        ].iloc[0]

        rk = perf[
            (perf["run"].eq(run_idx))
            & perf["scenario"].eq(
                "Random-K"
            )
        ].iloc[0]

        matched.append({
            "run": int(run_idx),
            "vr_k":
                int(vr["selected_client_count"]),
            "randomk_k":
                int(rk["selected_client_count"]),
            "same_k":
                int(vr["selected_client_count"])
                == int(rk["selected_client_count"]),
            "vr_total_optimizer_steps":
                int(vr["optimizer_steps"]),
            "randomk_total_optimizer_steps":
                int(rk["optimizer_steps"]),
            "exact_total_step_parity":
                int(vr["optimizer_steps"])
                == int(rk["optimizer_steps"]),
            "same_initial_weights":
                str(vr["initial_weights_sha256"])
                == str(rk["initial_weights_sha256"]),
        })

    matched = pd.DataFrame(matched)
    matched.to_csv(
        EXPERIMENT_ROOT /
        "tadp_vr_randomk_matched_control_audit.csv",
        index=False,
    )

    if not matched[
        [
            "same_k",
            "exact_total_step_parity",
            "same_initial_weights",
        ]
    ].all().all():
        raise RuntimeError(
            "CIFAR matched TADP-VR/Random-K audit failed."
        )

    pd.DataFrame(
        selection_rows
    ).drop_duplicates().to_csv(
        EXPERIMENT_ROOT /
        "selection_audit.csv",
        index=False,
    )

    perf = perf.sort_values(
        ["run", "scenario"]
    ).reset_index(drop=True)

    perf.to_csv(
        EXPERIMENT_ROOT /
        "performance_metrics.csv",
        index=False,
    )

    summary = summarize_runs_with_ci(
        perf
    )
    summary.to_csv(
        EXPERIMENT_ROOT /
        "performance_summary_mean_sd_ci95.csv",
        index=False,
    )

    paired = paired_vr_randomk_statistics(
        perf
    )
    paired.to_csv(
        EXPERIMENT_ROOT /
        "paired_vr_randomk_statistics.csv",
        index=False,
    )

    resource_cols = [
        "run",
        "seed",
        "scenario",
        "runtime_s",
        "energy_wh",
        "energy_kwh",
        "co2_kg",
        "communication_mb",
        "ram_peak_mb",
        "ram_delta_mb",
        "energy_cost_usd",
        "communication_cost_usd",
        "total_estimated_cost_usd",
        "optimizer_steps",
        "selected_client_count",
    ]

    perf[resource_cols].to_csv(
        EXPERIMENT_ROOT /
        "resource_metrics.csv",
        index=False,
    )

    policy = {
        "experiment_version":
            EXPERIMENT_VERSION,
        "scenarios":
            CIFAR_SCENARIOS,
        "scenario_count":
            len(CIFAR_SCENARIOS),
        "training_seeds":
            TRAINING_RUN_SEEDS,
        "number_of_runs":
            len(TRAINING_RUN_SEEDS),
        "fl_rounds":
            NUM_ROUNDS_FL,
        "local_epochs":
            LOCAL_EPOCHS,
        "batch_size":
            BATCH_SIZE,
        "learning_rate":
            LEARNING_RATE,
        "fedprox_mu":
            FEDPROX_MU,
        "dirichlet_alpha":
            DIRICHLET_ALPHA,
        "HPS_weights":
            WEIGHTS_PSCORE_DEFAULT,
        "HPS_review_threshold":
            GOOD_CUT,
        "HPS_direct_accept_threshold":
            HIGH_CUT,
        "global_cifar10_WAC":
            WAC_CIFAR10,
        "cifar10_dimension_WAC":
            CIFAR10_DIMENSION_WAC,
        "WAC_is_domain_policy_value":
            True,
        "client_review_metric":
            "Client Adequacy Score (CAS_i)",
        "dimension_min_floor":
            DIMENSION_MIN_FLOOR,
        "direct_auto_accept_rule":
            "critical score >= factor-specific adequacy minimum",
        "global_critical_wac":
            derive_global_critical_wac("cifar10"),
        "critical_factors":
            CRITICAL_FACTORS_BY_DOMAIN["cifar10"],
        "total_governance_factors":
            28,
        "DQ_train_only_factors":
            8,
        "GX_matched_k_count":
            len(ge_clients),
        "governance_frozen_across_runs":
            True,
        "TADP_VR_frozen_across_runs_and_rounds":
            True,
        "RandomK_frozen_across_runs_and_rounds":
            True,
        "RandomK_exact_step_parity":
            True,
        "official_test_semantics":
            "final evaluation only",
        "validation_semantics":
            "held outside client partition; not used for governance or tuning",
    }

    atomic_write_json(
        policy,
        EXPERIMENT_ROOT /
        "policy_and_experiment_design.json",
    )

    print_banner(
        "FINAL CIFAR PERFORMANCE SUMMARY — "
        "MEAN ± SD / 95% CI SAVED"
    )

    cols = [
        "scenario",
        "n_runs",
        "accuracy_mean",
        "accuracy_sd",
        "f1_macro_mean",
        "f1_macro_sd",
        "roc_auc_ovr_macro_mean",
        "roc_auc_ovr_macro_sd",
        "runtime_s_mean",
        "communication_mb_mean",
        "ram_peak_mb_mean",
    ]

    print(
        summary[
            [c for c in cols if c in summary.columns]
        ].to_string(index=False)
    )

    print_banner(
        "CIFAR TADP-VR vs RANDOM-K "
        "PAIRED STATISTICS"
    )
    print(paired.to_string(index=False))

    print_banner("FINAL CIFAR FAIL-CLOSED AUDITS")
    print("No-leakage audit: PASS")
    print("Within-run W0 parity: PASS")
    print("TADP-VR frozen across all runs/rounds: PASS")
    print("Random-K frozen across all runs/rounds: PASS")
    print("TADP-VR/Random-K same K: PASS")
    print("TADP-VR/Random-K exact optimizer-step parity: PASS")
    print(
        f"Completed {len(CIFAR_SCENARIOS)} scenarios x "
        f"{len(TRAINING_RUN_SEEDS)} runs."
    )

    return EXPERIMENT_ROOT


if __name__ == "__main__":
    finished_root = main()
    package_and_download_results(
        finished_root,
        EXPERIMENT_VERSION,
    )


Mounted at /content/drive
Persistent checkpoint root: /content/drive/MyDrive/TADP_CHECKPOINTS/TADP_EXPERIMENT_C_CIFAR10_v16_7_GXCORE_MATCHEDKFIX

TADP-CIFAR10-v16.7-FULL-NOLEAK-GXCORE-MATCHEDKFIX-CRITICALWAC-28F
Experimental training seeds: [42, 142, 242]
Each training seed executes ALL 7 CIFAR federated scenarios.
Total trained configurations: 21
FL rounds: 10
Local epochs: 1
Batch size: 128
Dirichlet alpha: 1.0
One-time evidence-assignment RNG seed: 3042 [not an experimental run seed]
One-time Random-K selection RNG seed: 4042 [not an experimental run seed]
Global CIFAR-10 reference WAC: 0.676111 (derived from 28 factor-specific adequacy ranks)
170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 26s 0us/step

NO-LEAKAGE AUDIT — MUST PASS
 train_test_overlap  test_rows_in_any_client  train_rows_missing_from_clients  client_rows_not_in_global_train  duplicate_train_rows_across_clients  pass  official_test_untouched_before_final_evaluation  validation_excluded_from_client_partition  normalization_

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmppz4se2yr' for ephemeral docs site


Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

...

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]

Calculating Metrics:   0%|          | 0/65 [00:00<?, ?it/s]


DOMAIN ADEQUACY POLICY — FULL WAC + CRITICAL WAC
dimension                      dimension_name  n_factors minimum_adequacy_ranks  dimension_policy_wac
     dim1                  Source Reliability          4                4,4,4,4              0.800000
     dim2             Data Quality and Health          8        3,3,3,3,3,3,3,3              0.600000
     dim3             Documentation Practices          4                3,3,4,3              0.650000
     dim4         Timeliness and Refresh Rate          3                  3,3,4              0.666667
     dim5 Regulatory and Compliance Alignment          5              3,3,3,3,4              0.640000
     dim6       Context and Usage Constraints          4                3,4,3,4              0.700000

GLOBAL CIFAR10 WAC (descriptive full-policy summary) = 0.676111
GLOBAL CIFAR10 CRITICAL WAC (automated Review threshold) = 0.750000

CRITICAL FACTORS — INDIVIDUAL ADEQUACY REQUIREMENTS
dimension                  factor  individual_adeq

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>